# Amazon ML Challenge 2026 — Business Entity Resolution

**Task.** Source 1 is a de-duplicated reference list of businesses (`entity_id`, `business_name`, `business_address`, `country`). Sources 2 and 3 are noisy feeds of the same kind of records (S2 shouts in ALL-CAPS and transliterates 15% of its names into Indic scripts, S3 abbreviates, spells states out, inserts aliases such as `X DBA Y`, moves legal tokens around, appends junk, drops addresses). For **every** Source-1 entity of the test set we must return **all** Source-2/Source-3 records that describe the same business — zero, one or many — and the set of candidates our blocking stage considered. The score is the F0.5 of each Source-1 entity's predicted set, macro-averaged over all Source-1 entities (a singleton scores 1.0 when we predict nothing for it and 0.0 when we predict anything), so precision counts four times more than recall.

**Pipeline (final design, derived from a statistical analysis of the training files):**

| stage | what happens | output |
|---|---|---|
| Load | dynamic file discovery, chunked TSV reading with `QUOTE_NONE`, `NULL`/`N/A` kept as noise tokens | raw frames (pyarrow strings) |
| Clean | alias prefixes, URLs, phone numbers, `(ID: …)` junk, brackets, dotted abbreviations, `&`↔`and`, repeated words, legal tokens, honorifics | tokens |
| Normalize | ordered `name_norm` for similarity, order-free `name_key` + `leet_key` for blocking, address components → state / geo / city / house / street / postcode, `addr_norm` / `addr_key` | `NormTable` per source |
| Block | nine hashed key families (exact name, name+geo, exact address, first token, house+street, 5-char prefix, Soundex, per-token, geo-less token), block caps, union with a hit mask, ranking, ≤ 50 candidates per entity | candidate pairs = `candidate_pairs.tsv` |
| Features | exactly 77 float32 pair features (string similarities, TF-IDF cosines, token statistics, address agreement, name frequencies, blocking hits, entity-relative ranks/gaps) | feature matrix |
| Model | LightGBM binary classifier trained on the blocked pairs of 120k training entities with early stopping on 40k validation entities (split **by entity**) | pair probabilities |
| Threshold | grid over thresholds and entity-relative margins, maximising validation macro F0.5 | decision rule |
| Output | `output/matching_results.tsv` and `output/candidate_pairs.tsv`, validated with the official rules | submission |

**How to run on Kaggle.** Attach the challenge data as a dataset (any folder layout; numeric upload prefixes such as `1790278321851_train_source1.tsv` are fine), use a CPU session with internet disabled, and *Run All*. The full run takes about 1.5–2 h: the Source-2 + Source-3 pool is always used in full, and `N_TRAIN_S1` / `N_VAL_S1` (120,000 / 40,000 Source-1 entities) trade time for accuracy. Setting the environment variable `ER_QUICK=1` runs the same code on 3,000 / 1,500 entities as a smoke test.

**Expected input files** (found recursively under `/kaggle/input`, or `ER_DATA_ROOT`, `./dataset`, `.`, `..`, `/mnt/user-data/uploads`):

| file | required? | what happens when absent |
|---|---|---|
| `train_source1.tsv`, `train_source2.tsv`, `train_source3.tsv` | required | the notebook stops with a clear error |
| `train_ground_truth.tsv` | required | the notebook stops: it provides labels, recall denominators, the transliteration dictionary and the threshold |
| `test_source1.tsv`, `test_source2.tsv`, `test_source3.tsv` | required for a submission, optional for training | training and evaluation still run; the validation split is written in the challenge format and clearly labelled as such |
| `utils/validate_submission.py` | optional | the notebook re-implements the same rules; the official script is run in addition when found |

No other files, external data, services, pretrained models or lookups are used (external lookups are grounds for disqualification). Country is an **open set**: the training data has US and India, the test data also has France; nothing filters, one-hot-encodes or asserts on the country value.

In [ ]:
# =====================================================================================
# Cell 1 - setup: imports, optional libraries, seeds, configuration, logging, progress
# =====================================================================================
import os, re, sys, gc, csv, glob, json, math, time, random, unicodedata, warnings, subprocess
from collections import Counter, defaultdict
import multiprocessing
from multiprocessing import Pool

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import matplotlib
try:                                   # inline plots inside Jupyter, headless Agg elsewhere
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 80)

# ---- optional libraries: every one of them has a working fallback further down --------------
try:
    import rapidfuzz
    from rapidfuzz import fuzz as rf_fuzz, process as rf_process
    from rapidfuzz.distance import JaroWinkler as rf_JW, Levenshtein as rf_Lev
    HAVE_RF = True
except Exception:                       # pure-Python difflib / Jaro-Winkler / Levenshtein fallbacks
    HAVE_RF = False
try:
    import lightgbm as lgb
    HAVE_LGB = True
except Exception:                       # sklearn HistGradientBoostingClassifier fallback
    HAVE_LGB = False
try:
    import pyarrow as pa
    HAVE_PA = True
except Exception:                       # object-dtype fallback for the normalized string columns
    HAVE_PA = False
try:
    from unidecode import unidecode
    HAVE_UNIDECODE = True
except Exception:                       # Indic tokens are handled by the learned transliteration map alone
    HAVE_UNIDECODE = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CONFIG = dict(
    DATA_ROOT=os.environ.get('ER_DATA_ROOT'),                                   # optional override
    OUTPUT_DIR='/kaggle/working/output' if os.path.exists('/kaggle/working') else 'output',
    ARTIFACT_DIR='/kaggle/working/artifacts' if os.path.exists('/kaggle/working') else 'artifacts',
    N_TRAIN_S1=120_000, N_VAL_S1=40_000,      # Source-1 entities sampled for train / validation
    MAX_CAND_PER_S1=50,
    BLOCK_CAPS=dict(n=400, nc=150, a=200, g1=300, hs=120, p5=300, ph=300, tk=150, tk0=60),
    N_JOBS=max(1, min(4, os.cpu_count() or 1)),
    CHUNK_S1=40_000,                          # Source-1 entities per blocking / inference chunk
    PAIR_CHUNK=250_000,                       # pairs per feature-building chunk (entity-aligned)
    TFIDF_FIT_ROWS=500_000, LGB_ROUNDS=3000, SEED=SEED,
    QUICK=os.environ.get('ER_QUICK', '0') == '1',   # smoke-test mode (3,000 / 1,500 entities)
)
if CONFIG['QUICK']:                           # smoke test: same code path, a fraction of the work
    CONFIG.update(N_TRAIN_S1=3_000, N_VAL_S1=1_500, LGB_ROUNDS=300, TFIDF_FIT_ROWS=60_000)

OUTPUT_DIR, ARTIFACT_DIR = CONFIG['OUTPUT_DIR'], CONFIG['ARTIFACT_DIR']
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)
N_JOBS = CONFIG['N_JOBS']
T_NOTEBOOK_START = time.time()


def log(msg):
    """Timestamped, flushed log line (Kaggle buffers stdout otherwise)."""
    print(f"[{time.strftime('%H:%M:%S')} +{fmt_secs(time.time() - T_NOTEBOOK_START)}] {msg}", flush=True)


def fmt_secs(s):
    s = int(s)
    return (f"{s//3600}h{(s%3600)//60:02d}m{s%60:02d}s" if s >= 3600
            else (f"{s//60}m{s%60:02d}s" if s >= 60 else f"{s}s"))


class Progress:
    """Prints one line each time another 1% of `total` units completes:
       [label]  37.00% | last 1%: 2.1s | elapsed: 1m18s | ETA: 2m12s"""
    def __init__(self, total, label, step_pct=1.0):
        self.total = max(1, int(total)); self.label = label; self.step = step_pct
        self.t0 = time.time(); self.t_last = self.t0; self.done = 0; self.next_pct = step_pct
    def update(self, n=1):
        self.done += n; pct = 100.0 * self.done / self.total
        if pct + 1e-9 >= self.next_pct or self.done >= self.total:
            now = time.time(); elapsed = now - self.t0
            eta = elapsed / max(self.done, 1) * (self.total - self.done)
            print(f"[{self.label}] {min(pct, 100):6.2f}% | last {self.step:g}%: {now - self.t_last:6.1f}s | "
                  f"elapsed: {fmt_secs(elapsed)} | ETA: {fmt_secs(eta)}", flush=True)
            self.t_last = now
            while self.next_pct <= pct: self.next_pct += self.step
    def close(self):
        if self.done < self.total: self.done = self.total; self.update(0)


def pool_imap(func, jobs, n_jobs, label, timeout=1800):
    """Ordered imap over a fork Pool of top-level `func`.  gc.freeze() keeps the forked workers from
    copy-on-write-duplicating the parent's heap when their garbage collector runs; a job that produces no
    result within `timeout` seconds means a worker was killed (usually out of memory) and raises instead of
    hanging forever.  Serial when n_jobs == 1 or fork is unavailable."""
    if n_jobs > 1:
        try:
            ctx = multiprocessing.get_context('fork')
        except ValueError:
            ctx = None
        if ctx is not None:
            gc.collect(); gc.freeze()
            try:
                with ctx.Pool(n_jobs, initializer=gc.disable) as pool:
                    it = pool.imap(func, jobs, chunksize=1)
                    while True:
                        try:
                            yield it.next(timeout=timeout)
                        except StopIteration:
                            break
                        except multiprocessing.TimeoutError:
                            raise RuntimeError(f"{label}: a worker produced no result for {timeout}s - it was probably "
                                               f"killed (out of memory). Rerun with more memory or N_JOBS=1.")
            finally:
                gc.unfreeze()
            return
    for job in jobs:
        yield func(job)


def progress_for(n_units, label):
    """A Progress whose step is 1% when the step is divisible into >= 100 units, otherwise one line per unit."""
    n_units = max(1, int(n_units))
    return Progress(n_units, label, step_pct=1.0 if n_units >= 100 else 100.0 / n_units)


log(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | "
    f"rapidfuzz={'yes ' + rapidfuzz.__version__ if HAVE_RF else 'NO (difflib fallback)'} | "
    f"lightgbm={'yes ' + lgb.__version__ if HAVE_LGB else 'NO (HistGradientBoosting fallback)'} | "
    f"pyarrow={'yes ' + pa.__version__ if HAVE_PA else 'NO (object dtype)'} | "
    f"unidecode={'yes' if HAVE_UNIDECODE else 'no'}")
log(f"N_JOBS={N_JOBS} | QUICK={CONFIG['QUICK']} | OUTPUT_DIR={os.path.abspath(OUTPUT_DIR)} | "
    f"ARTIFACT_DIR={os.path.abspath(ARTIFACT_DIR)}")

## 1. Loading

`find_file` discovers each file dynamically (exact name or `*name` for prefixed uploads) and prints the resolved paths. Files are read in 250k-row chunks with a 1% progress tracker after a fast line count. Two reader settings matter for this data: `quoting=csv.QUOTE_NONE`, because addresses contain commas and stray quotes and the files are written unquoted, and `keep_default_na=False` with `na_values=['']`, because the literal strings `NULL` / `N/A` / `None` appear inside addresses as injected noise tokens (3.4% of S2/S3) — only an empty cell is missing. The ground truth is mandatory. Sources 2 and 3 are concatenated into one candidate pool that is always used in full.

In [ ]:
# =====================================================================================
# Cell 2 - dynamic file discovery and chunked TSV loading
# =====================================================================================
SEARCH_ROOTS = [r for r in [CONFIG['DATA_ROOT'], '/kaggle/input', './dataset', '.', '..', '/mnt/user-data/uploads'] if r]


def find_file(name):
    """Locate `name` (or a prefixed upload such as 1790278321851_train_source1.tsv) under the search roots.
    Roots are tried in order; within the first root that has a hit the shortest path wins."""
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        hits = []
        for pat in (name, '*' + name):
            hits += glob.glob(os.path.join(root, '**', pat), recursive=True)
            hits += glob.glob(os.path.join(root, pat))
        hits = sorted({os.path.abspath(h) for h in hits if os.path.isfile(h)}, key=lambda p: (len(p), p))
        if hits:
            return hits[0]
    return None


FILE_ROLES = [
    ('train_source1.tsv',      'REQUIRED', 'query side for training / validation, ground-truth keys'),
    ('train_source2.tsv',      'REQUIRED', 'candidate pool (used in full)'),
    ('train_source3.tsv',      'REQUIRED', 'candidate pool (used in full)'),
    ('train_ground_truth.tsv', 'REQUIRED', 'labels, recall denominators, transliteration dictionary, threshold'),
    ('test_source1.tsv',       'optional', 'rows of both output files (validation split is written when absent)'),
    ('test_source2.tsv',       'optional', 'test candidate pool / universe of valid ids'),
    ('test_source3.tsv',       'optional', 'test candidate pool / universe of valid ids'),
    ('validate_submission.py', 'optional', 'official validator (run when found; the notebook re-implements its rules)'),
]
PATHS = {name: find_file(name) for name, _, _ in FILE_ROLES}
print(pd.DataFrame([(n, r, PATHS[n] or '-- not found --', d) for n, r, d in FILE_ROLES],
                   columns=['file', 'required', 'resolved path', 'role']).to_string(index=False))
for name, req, _ in FILE_ROLES:
    if req == 'REQUIRED':
        assert PATHS[name], (f"{name} is required but was not found under {SEARCH_ROOTS}. "
                             f"Attach the challenge data as a Kaggle dataset (any folder layout).")
    elif PATHS[name] is None:
        log(f"WARNING: optional file {name} not found")
HAVE_TEST = all(PATHS[f'test_source{i}.tsv'] for i in (1, 2, 3))
if not HAVE_TEST:
    log("WARNING: test files missing -> the validation split will be written in the challenge format instead")

REQUIRED_COLS = ['entity_id', 'business_name', 'business_address', 'country']


def count_lines(path):
    """Fast newline count so the chunked reader can report an exact percentage."""
    n = 0
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            n += block.count(b'\n')
    return n


def read_tsv(path, label=None):
    """Chunked TSV reader with 1% progress.
    QUOTE_NONE: addresses contain commas and stray quotes and the files are written unquoted, so a quote
    character must never start a quoted field.  keep_default_na=False + na_values=['']: the literal strings
    NULL / N/A / None are injected NOISE TOKENS inside addresses (~3.4% of S2/S3), not missing values; only an
    empty cell is missing."""
    label = label or os.path.basename(path)
    t0 = time.time()
    total = max(1, count_lines(path) - 1)
    prog = Progress(total, f"load {label}")
    parts = []
    # pyarrow-backed strings keep 12.5M rows in ~1 GB instead of ~4 GB of Python objects (object dtype fallback)
    dtype = 'string[pyarrow]' if HAVE_PA else str
    reader = pd.read_csv(path, sep='\t', dtype=dtype, quoting=csv.QUOTE_NONE, keep_default_na=False,
                         na_values=[''], encoding='utf-8', encoding_errors='replace', on_bad_lines='warn',
                         chunksize=250_000)
    for chunk in reader:
        parts.append(chunk)
        prog.update(len(chunk))
    prog.close()
    df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=REQUIRED_COLS)
    del parts
    df.columns = [c.strip() for c in df.columns]
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    assert not missing, f"{path}: missing columns {missing}; found {list(df.columns)}"
    df = df[REQUIRED_COLS].copy()
    df['entity_id'] = df['entity_id'].fillna('').str.strip()
    df['business_name'] = df['business_name'].fillna('')
    # business_address keeps its missing values (NA) - the only genuinely missing field in the data
    df['country'] = df['country'].fillna('').str.strip()
    log(f"loaded {label}: {len(df):,} rows in {time.time() - t0:.1f}s")
    return df


class IdIndex:
    """entity_id -> row position through sorted uint64 hashes (pd.util.hash_array).  A pandas Index over
    10M pyarrow strings materialises every id as a Python object plus a hash table (~1.5 GB); this keeps
    12 bytes per id.  Hash uniqueness is verified at build time (pandas Index fallback on a collision)."""
    def __init__(self, series):
        self.n = len(series)
        hashes = np.empty(self.n, dtype=np.uint64)
        for a in range(0, self.n, 1_000_000):
            hashes[a:a + 1_000_000] = pd.util.hash_array(np.asarray(series.iloc[a:a + 1_000_000].tolist(), dtype=object))
        self.order = np.argsort(hashes, kind='stable').astype(np.int32)
        self.sorted = hashes[self.order]
        self.fallback = None
        if self.n and (np.diff(self.sorted) == 0).any():        # duplicated ids or (astronomically rare) collisions
            self.fallback = pd.Index(np.asarray(series.tolist(), dtype=object))
    def get_indexer(self, ids):
        ids = list(ids) if not isinstance(ids, (list, np.ndarray)) else ids
        if len(ids) == 0:
            return np.zeros(0, dtype=np.int64)
        if self.fallback is not None:
            return self.fallback.get_indexer(np.asarray(ids, dtype=object))
        h = pd.util.hash_array(np.asarray(ids, dtype=object))
        i = np.minimum(np.searchsorted(self.sorted, h), max(self.n - 1, 0))
        found = (self.sorted[i] == h) if self.n else np.zeros(len(ids), bool)
        return np.where(found, self.order[i], -1).astype(np.int64)
    def get_loc(self, one_id):
        return int(self.get_indexer([one_id])[0])
    def __len__(self):
        return self.n


def parse_ground_truth(path):
    """train_ground_truth.tsv -> {source1_entity_id: matched ids (unique, tuple)}; empty for singletons."""
    if not path or not os.path.isfile(path):
        raise FileNotFoundError("train_ground_truth.tsv is mandatory: it provides the labels, the recall "
                                "denominators, the transliteration dictionary and the decision threshold.")
    t0 = time.time()
    g = pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE,
                    encoding='utf-8', encoding_errors='replace')
    g.columns = [c.strip() for c in g.columns]
    assert {'source1_entity_id', 'matched_entity_ids'} <= set(g.columns), f"unexpected GT columns {list(g.columns)}"
    gt = {}
    for s1, ids in zip(g['source1_entity_id'].tolist(), g['matched_entity_ids'].tolist()):
        s1 = s1.strip()
        gt[s1] = tuple(dict.fromkeys(x.strip() for x in ids.split(',') if x.strip())) if ids and ids.strip() else ()
    log(f"ground truth: {len(gt):,} Source-1 entities, {sum(len(v) for v in gt.values()):,} matched ids "
        f"({time.time() - t0:.1f}s)")
    return gt


T_LOAD = time.time()
S1 = read_tsv(PATHS['train_source1.tsv'], 'train_source1')
S2 = read_tsv(PATHS['train_source2.tsv'], 'train_source2')
S3 = read_tsv(PATHS['train_source3.tsv'], 'train_source3')
POOL = pd.concat([S2, S3], ignore_index=True)
N_S2, N_S3 = len(S2), len(S3)
del S2, S3
gc.collect()
GT = parse_ground_truth(PATHS['train_ground_truth.tsv'])
S1_POS = IdIndex(S1['entity_id'])          # entity_id -> row position (used by EDA, GT pairs, splits)
POOL_POS = IdIndex(POOL['entity_id'])
log(f"sizes: S1={len(S1):,}  POOL(S2+S3)={len(POOL):,} (S2={N_S2:,}, S3={N_S3:,})  "
    f"total={len(S1) + len(POOL):,}  | load time {fmt_secs(time.time() - T_LOAD)}")

## 2. EDA / data-quality checks

Measured on the training files (numbers printed below come from the loaded data): S1 2,206,821 rows, S2 5,034,616, S3 5,285,603 (12.5M in total; US ≈ 60%, India ≈ 40%). Names are never missing; addresses are missing only in S2 (3.36%) and S3 (3.33%). Exact full-record duplicates (0.71%) exist only inside S2/S3. The noise-pattern table quantifies the corruption types that drive the cleaning rules: ALL-CAPS names (S2 26.8%), Indic-script names (S2 15.2%, S3 11.5%, S1 0%), URL/phone/bracket junk, alias prefixes (S3 only), domain-style names (`strategichoovercom`), digit-for-letter typos, spelled-out or Indic-script states, `NULL`/`N/A` fillers, `PO Box` insertions, zero-padded or punctuated house numbers and `St → Saint` mis-expansions. The ground-truth block shows the distribution of matches per Source-1 entity, the singleton share (verified on the real ground truth, not assumed) and the S2/S3 split of the matched ids. Six true pairs are printed raw so the corruption types are visible.

In [ ]:
# =====================================================================================
# Cell 3 - EDA / data-quality checks (all numbers computed from the loaded files)
# =====================================================================================
T_EDA = time.time()
_S2_MASK = np.arange(len(POOL)) < N_S2       # POOL = S2 rows followed by S3 rows


def str_contains(series, pattern):
    """series.str.contains(regex) with a pure-Python `re` fallback: pyarrow-backed strings route regexes
    through RE2, which rejects some escapes (look-arounds, \\uXXXX)."""
    try:
        out = series.str.contains(pattern, regex=True)
        return out.fillna(False).to_numpy(dtype=bool)
    except Exception:
        rx = re.compile(pattern)
        return np.fromiter((bool(rx.search(x)) if isinstance(x, str) else False for x in series.tolist()),
                           dtype=bool, count=len(series))


# ---- (a) overview per source --------------------------------------------------------------------
def overview(df, label):
    name_missing = int((df['business_name'].fillna('').str.strip() == '').sum())
    addr = df['business_address']
    addr_missing = int(addr.isna().sum() + (addr.fillna('x').str.strip() == '').sum())
    countries = df['country'].value_counts().head(6)
    return dict(source=label, rows=len(df), missing_names=name_missing,
                missing_addresses=f"{addr_missing:,} ({100.0 * addr_missing / max(1, len(df)):.2f}%)",
                distinct_names=int(df['business_name'].nunique()), distinct_addresses=int(addr.nunique()),
                countries=', '.join(f"{k}:{v:,}" for k, v in countries.items()),
                id_prefixes=', '.join(sorted(df['entity_id'].str.slice(0, 3).unique().tolist())))


OVERVIEW = pd.DataFrame([overview(S1, 'S1'), overview(POOL[_S2_MASK], 'S2'), overview(POOL[~_S2_MASK], 'S3')])
print("=== (a) overview per source ===")
print(OVERVIEW.to_string(index=False))

# ---- (b) noise patterns on a 200k-row sample per source -----------------------------------------
NOISE_PATTERNS = [
    ('all-caps name',              r'^[^a-z]*$', 'name'),
    ('non-ASCII (Indic script)',   r'[^\x00-\x7F]', 'name'),
    ("'| www' / URL appended",     r'(\||www\.|https?:)', 'name'),
    ('alias prefix',               r'(?i)\b(d/?b/?a|doing business as|f/?k/?a|t/a|formerly|a/?k/?a)\b', 'name'),
    ('domain-style name',          r'(?i)^[a-z0-9]+\.?com$', 'name'),
    ('digit inside word',          r'[a-zA-Z][0-9][a-zA-Z]', 'name'),
    ('phone appended',             r'\b\d{7,}\b', 'name'),
    ('double spaces',              r'  ', 'name'),
    ('all-caps address',           r'^[^a-z]*$', 'address'),
    ('state spelled out (US)',     r'(?i),\s*(texas|new york|north carolina|ohio|illinois|california)\s*$', 'address'),
    ('NULL/N/A filler',            r'(?i)\b(null|n/a|none)\b', 'address'),
    ('PO Box inserted',            r'(?i)\bp\.?o\.? box\b', 'address'),
    ('leading-zero house no.',     r'(?i)(^|, ?)0[0-9]+ ', 'address'),
    ('house no. punctuation',      r'#{2}|[0-9]+\. |[0-9]+- ', 'address'),
    ('Saint',                      r'(?i)\bsaint\b', 'address'),
]
_rs = np.random.RandomState(SEED)
SAMPLES = {'S1': S1.sample(min(200_000, len(S1)), random_state=_rs),
           'S2': POOL[_S2_MASK].sample(min(200_000, int(_S2_MASK.sum())), random_state=_rs),
           'S3': POOL[~_S2_MASK].sample(min(200_000, int((~_S2_MASK).sum())), random_state=_rs)}
rows = []
for pname, pat, col in NOISE_PATTERNS:
    r = {'pattern': pname, 'field': col}
    for src, smp in SAMPLES.items():
        ser = smp['business_name'] if col == 'name' else smp['business_address'].fillna('')
        r[src] = f"{100.0 * str_contains(ser, pat).mean():.2f}%"
    rows.append(r)
NOISE_TABLE = pd.DataFrame(rows)
print("\n=== (b) noise patterns (% of a 200k-row sample per source) ===")
print(NOISE_TABLE.to_string(index=False))

# ---- (c) ground-truth statistics ------------------------------------------------------------------
n_match = np.array([len(v) for v in GT.values()], dtype=np.int64)
bins = np.minimum(n_match, 6)
dist = pd.Series(np.bincount(bins, minlength=7), index=['0', '1', '2', '3', '4', '5', '6+'])
SINGLETON_SHARE = float((n_match == 0).mean())
print("\n=== (c) ground truth ===")
print(pd.DataFrame({'entities': dist, 'share': (dist / dist.sum()).map(lambda x: f"{100 * x:.2f}%")}).to_string())
print(f"singleton share (entities with zero matches): {100 * SINGLETON_SHARE:.2f}%  | mean matches "
      f"{n_match.mean():.3f} | max {n_match.max()} | S1 entities in GT: {len(GT):,} of {len(S1):,}")
all_gt_ids = [x for v in GT.values() for x in v]
pref = Counter(x[:2] for x in all_gt_ids)
print(f"matched ids by source: S2={pref.get('S2', 0):,}  S3={pref.get('S3', 0):,}  other={sum(v for k, v in pref.items() if k not in ('S2', 'S3')):,}")
_gt_sample = _rs.choice(np.array(all_gt_ids, dtype=object), size=min(200_000, len(all_gt_ids)), replace=False)
print(f"share of sampled GT ids that exist in S2/S3: {100.0 * (POOL_POS.get_indexer(_gt_sample) >= 0).mean():.3f}%")
del all_gt_ids, _gt_sample

# ---- (d) six true pairs side by side -----------------------------------------------------------------
print("\n=== (d) six true pairs (raw): S1 record, then its matched records ===")
_shown = 0
for s1_id in list(GT.keys()):
    ids = GT[s1_id]
    if not ids:
        continue
    i = S1_POS.get_loc(s1_id)
    print(f"\n[{s1_id}] {S1['business_name'].iat[i]} | {S1['business_address'].iat[i]} | {S1['country'].iat[i]}")
    for pid in sorted(ids)[:3]:
        j = POOL_POS.get_loc(pid)
        if j >= 0:
            print(f"    -> [{pid}] {POOL['business_name'].iat[j]} | {POOL['business_address'].iat[j]}")
    _shown += 1
    if _shown >= 6:
        break

# ---- (e) two small charts ---------------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].bar(dist.index, dist.values, color='#4c72b0'); ax[0].set_title('GT: matches per Source-1 entity'); ax[0].set_ylabel('entities')
cc = S1['country'].value_counts().head(8)
ax[1].bar(cc.index.astype(str), cc.values, color='#dd8452'); ax[1].set_title('Source-1 country (open set)'); ax[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.savefig(os.path.join(ARTIFACT_DIR, 'eda_charts.png'), dpi=90); plt.show(); plt.close(fig)
del SAMPLES
gc.collect()
log(f"EDA finished in {fmt_secs(time.time() - T_EDA)}")

## 3. Cleaning rules and the evidence behind them

**Names**
- Keep only the text before the first `|`, remove URLs and 7+ digit numbers: `Procopio … Services | www.procopio.com`, `Pacific Program LLC - 7418920210` are junk appendices, never name content.
- Remove `(ID: 75168)` style tags (seen appended in S2/S3).
- `&` → `and` before tokenising (the two forms alternate freely between sources).
- Latin accents are stripped (`Bureau óf Párks` → `bureau of parks`) **but only when every letter is Latin**: NFKD on an Indic string separates the vowel signs from their consonants and destroys the word.
- Alias markers `DBA`, `doing business as`, `f/k/a`, `t/a`, `formerly`, `a/k/a`, `trading as` (S3, ~2% of names): in 99.5% of true pairs the real name is the part **after** the marker (`Avizephdrex DBA LUK Energetics Inc` → `luk energetics`).
- Indian honorific prefixes (`Smt`, `Shri`, `Sri`, `Dr`, `M/s`) added by S2/S3 are dropped when something remains.
- Dotted abbreviations lose their dots (`L.L.C.` → `llc`), apostrophes are removed **without** a space (`Orelee's` → `orelees`), every other punctuation mark becomes a space (brackets around real words are removed, the words are kept: `[Llc] Optimal Superior Hudson,` → `optimal superior hudson`). The punctuation class covers ASCII and common Unicode punctuation only, so letters of every script survive.
- Domain-shaped names (`strategichoovercom`, `wilfordhancock.com`; ~3.5% of S2/S3, none in S1): the trailing `.com`/`com` is cut **only when the whole name is domain-shaped** — S1 contains genuine businesses such as `earnosethroat.com`, and `Franklin` or `Costco` merely end in a TLD.
- Consecutive repeated words collapse (`Creative Wireless Wireless Networks`), legal tokens (`inc`, `llc`, `ltd`, `pvt`, `private`, `corp`, `co`, `the`, `sarl`, `sas`, `gmbh`, …) are removed wherever they sit (`Inc Tyler Cleaning`, `Private Rex Consultants Ltd`): stripping them does not increase false merges (same-state agreement 84.9% vs 82.6% baseline). If that empties the name, all tokens are kept.
- Digit-for-letter typos (`C0rrine`, `6reendom`, 31% of single-character typos) are **not** overwritten (`3M`, `B2B` are real): a second `leet_key` is kept next to `name_key`.
- `name_norm` keeps the token order (string similarities); `name_key` is the sorted unique token set (blocking / equality). Token-sorting slightly raises the false-merge risk (77.6% same-state agreement), which is why order-free keys are used for **blocking** while the ordered string feeds the **features**.

**Addresses**
- Split on commas; drop empty / `NULL` / `None` / `N/A` components (S3 fillers), `PO Box …` insertions and `<NULL>` tokens.
- The state is recognised **at component level only** (`OH`, `Texas`, `Maharashtra`, `महाराष्ट्र`, `MH`) using the US and Indian tables — never at token level (`Washington Avenue` is not a state). An explicit 2-letter code wins over a spelled-out name; otherwise the last mapped component wins. Countries without a table (France) simply get no state.
- House-number prefixes (`H.No`, `Plot No`, `Flat No`, `Door No`, `Nº`, `#`, `##`) are removed, leading zeros are stripped from digit-only tokens (`05205` → `5205`), street types and directions are abbreviated (`Road` → `rd`, `Saint`/`Street` → `st`, French `R.`/`BD`/`ALL.`/`CRS` → `rue`/`blvd`/`allee`/`cours`), ordinal words become ordinals (`Eighth` → `8th`), historical Indian city names are mapped (`Poona` → `pune`, `Bombay` → `mumbai`, `Bengaluru` → `bangalore`, …).
- Unit designators (`Unit C`, `Apt 4B`, `Suite 200`, `Fl 0`, `2nd Floor`) are removed together with their number; Indian `Flat No 202` / `Shop 5` keep the number because it carries real information. Place words (`City`, `CDP`, `County`, `Township`, `Near`, `Opp`, …) are dropped.
- Postcodes are recognised only as 5–6 digit tokens in components with ≤ 2 tokens (`75002 Paris`, a standalone `69006`) or 6-digit tokens ending a component (Indian PINs): a 5-digit token inside `34184 Tanglewood Pass` is a US house number.
- `house` = the first digit-only token (`1098d` → `1098`), `street` = the first alphabetic token after it that is not a unit word, place word, street type or article, `city` = the last digit-free component. `geo`, the open-set blocking geography, is the state, else the postcode prefix, else the last words of the last two digit-free components (French addresses carry a region **or** a département that differs between sources: `Bordeaux, Nouvelle-Aquitaine` vs `MERIGNAC, Gironde`).
- `addr_norm` keeps the tokens in order, `addr_key` is the sorted unique token set (S1 shuffles comma components: `OH, Columbus, 5559 Orville Avenue`). Numeric tokens agree in only 36% of same-name true pairs, so house-number agreement is a soft feature, never a filter.

**Never do**: never delete non-ASCII text; never strip accents from non-Latin strings; never map state names at token level; never strip `.com` unless the whole name is domain-shaped; never overwrite digits with letters (keep a second leet key instead); never delete bracketed words (only the brackets).

In [ ]:
# =====================================================================================
# Cell 5 - normalization tables, regexes and functions
# =====================================================================================
US_STATES = {'alabama':'al','alaska':'ak','arizona':'az','arkansas':'ar','california':'ca','colorado':'co',
 'connecticut':'ct','delaware':'de','florida':'fl','georgia':'ga','hawaii':'hi','idaho':'id','illinois':'il',
 'indiana':'in','iowa':'ia','kansas':'ks','kentucky':'ky','louisiana':'la','maine':'me','maryland':'md',
 'massachusetts':'ma','michigan':'mi','minnesota':'mn','mississippi':'ms','missouri':'mo','montana':'mt',
 'nebraska':'ne','nevada':'nv','new hampshire':'nh','new jersey':'nj','new mexico':'nm','new york':'ny',
 'north carolina':'nc','north dakota':'nd','ohio':'oh','oklahoma':'ok','oregon':'or','pennsylvania':'pa',
 'rhode island':'ri','south carolina':'sc','south dakota':'sd','tennessee':'tn','texas':'tx','utah':'ut',
 'vermont':'vt','virginia':'va','washington':'wa','west virginia':'wv','wisconsin':'wi','wyoming':'wy',
 'district of columbia':'dc','washington dc':'dc','washington d.c.':'dc','puerto rico':'pr'}
US_CODES = set(US_STATES.values())
IN_STATES = {'maharashtra':'mh','delhi':'dl','new delhi':'dl','uttar pradesh':'up','karnataka':'ka',
 'tamil nadu':'tn','tamilnadu':'tn','gujarat':'gj','west bengal':'wb','telangana':'tg','ts':'tg',
 'andhra pradesh':'ap','haryana':'hr','rajasthan':'rj','kerala':'kl','keralam':'kl','bihar':'br',
 'madhya pradesh':'mp','punjab':'pb','odisha':'od','orissa':'od','or':'od','jharkhand':'jh','chhattisgarh':'cg',
 'chattisgarh':'cg','uttarakhand':'uk','uttaranchal':'uk','himachal pradesh':'hp','assam':'as','goa':'ga',
 'jammu and kashmir':'jk','jammu & kashmir':'jk','chandigarh':'ch','puducherry':'py','pondicherry':'py',
 'manipur':'mn','meghalaya':'ml','mizoram':'mz','nagaland':'nl','sikkim':'sk','tripura':'tr',
 'arunachal pradesh':'ar','andaman and nicobar islands':'an','dadra and nagar haveli':'dn','daman and diu':'dd',
 'lakshadweep':'ld','ladakh':'la',
 'महाराष्ट्र':'mh','दिल्ली':'dl','नई दिल्ली':'dl','उत्तर प्रदेश':'up','कर्नाटक':'ka','ಕರ್ನಾಟಕ':'ka','तमिलनाडु':'tn',
 'தமிழ்நாடு':'tn','गुजरात':'gj','ગુજરાત':'gj','पश्चिम बंगाल':'wb','পশ্চিমবঙ্গ':'wb','तेलंगाना':'tg','తెలంగాణ':'tg',
 'आंध्र प्रदेश':'ap','ఆంధ్రప్రదేశ్':'ap','हरियाणा':'hr','राजस्थान':'rj','केरल':'kl','കേരളം':'kl','बिहार':'br',
 'मध्य प्रदेश':'mp','पंजाब':'pb','ਪੰਜਾਬ':'pb','ओडिशा':'od','ଓଡ଼ିଶା':'od','झारखंड':'jh','छत्तीसगढ़':'cg','उत्तराखंड':'uk',
 'हिमाचल प्रदेश':'hp','असम':'as','অসম':'as','गोवा':'ga','जम्मू और कश्मीर':'jk','चंडीगढ़':'ch','पुडुचेरी':'py'}
IN_CODES = set(IN_STATES.values())
# Country is an OPEN SET: these tables are consulted only when the record's country has one.  Any other
# country (France in the test set) has no table and goes through the postcode / city fallbacks below.
COUNTRY_STATES = {'us': (US_STATES, US_CODES), 'usa': (US_STATES, US_CODES), 'united states': (US_STATES, US_CODES),
                  'united states of america': (US_STATES, US_CODES),
                  'india': (IN_STATES, IN_CODES), 'in': (IN_STATES, IN_CODES)}
# Spelling variants of the same country must land in the same blocking partition; unknown values pass through.
COUNTRY_CANON = {'usa': 'us', 'united states': 'us', 'united states of america': 'us', 'u.s.': 'us', 'u.s.a.': 'us',
                 'in': 'india', 'ind': 'india'}


def canon_country(c):
    c = (c or '').strip().lower()
    return COUNTRY_CANON.get(c, c)


ABBR = {'street':'st','saint':'st','avenue':'ave','av':'ave','road':'rd','drive':'dr','lane':'ln','lne':'ln','court':'ct',
 'boulevard':'blvd','place':'pl','circle':'cir','parkway':'pkwy','highway':'hwy','trail':'trl','terrace':'ter','square':'sq',
 'north':'n','south':'s','east':'e','west':'w','northeast':'ne','northwest':'nw','southeast':'se','southwest':'sw',
 'apartment':'apt','suite':'ste','building':'bldg','mount':'mt','fort':'ft','point':'pt','junction':'jct','extension':'extn',
 'ext':'extn','crossing':'xing','first':'1st','second':'2nd','third':'3rd','fourth':'4th','fifth':'5th','sixth':'6th',
 'seventh':'7th','eighth':'8th','ninth':'9th','tenth':'10th','floor':'fl','flr':'fl','number':'no','num':'no',
 'opposite':'opp','nr':'near','avenida':'ave','strasse':'str','heights':'hts','township':'twp','plaza':'plz',
 'centre':'center','ctr':'center',
 # French street types as abbreviated by S2/S3 ('63 R. DE DIEPPE', '51 BLVD DE REIMS', '28 ALL. MARCEL LEGOUEY',
 # '254 CRS DE L'YSER', '2 IMP DOCTEUR GUERIN'); test data contains France, which has no state table
 'r':'rue','bd':'blvd','bld':'blvd','boul':'blvd','crs':'cours','imp':'impasse','all':'allee','chem':'chemin',
 'rte':'route','fbg':'faubourg','qu':'quai'}
# street-type words and articles are skipped when locating the street NAME ('23 Boulevard de la Renaissance' ->
# 'renaissance', '1791 Harris Road' -> 'harris'); the house+street blocking key needs the name, not the type
STREET_SKIP = {'st','ave','rd','dr','ln','ct','blvd','pl','cir','pkwy','hwy','trl','ter','sq','way','rue','cours','impasse',
 'allee','chemin','route','faubourg','quai','avenue','street','road','boulevard','place','marg','nagar','main','cross',
 'de','du','des','la','le','les','l','d','the','of','and','bis','ter','old','new'}
HONORIFIC_RE = re.compile(r'^\s*(?:m/s\.?|messrs\.?|smt\.?|shri|shree|sri|dr\.?|mr\.?|mrs\.?|ms\.?)\s+', re.I)
ID_RE = re.compile(r'\(\s*id\s*:?\s*[\w-]+\s*\)', re.I)      # 'Enertia Vector (ID: 75168)'
CITY_SYN = {'bombay':'mumbai','calcutta':'kolkata','madras':'chennai','poona':'pune','bengaluru':'bangalore','gurugram':'gurgaon',
 'trivandrum':'thiruvananthapuram','baroda':'vadodara','mysuru':'mysore','belagavi':'belgaum','prayagraj':'allahabad',
 'cochin':'kochi','vizag':'visakhapatnam','vishakhapatnam':'visakhapatnam','panjim':'panaji','simla':'shimla',
 'cawnpore':'kanpur','benares':'varanasi','banaras':'varanasi','mangaluru':'mangalore','hubballi':'hubli',
 'tumakuru':'tumkur','shivamogga':'shimoga','ballari':'bellary','vijayapura':'bijapur','kalaburagi':'gulbarga',
 'puducherry':'pondicherry','secunderabad':'hyderabad'}
LEGAL_TOKENS = {'inc','incorporated','llc','ltd','limited','pvt','private','llp','lp','corp','corporation','co','company','plc',
 'pc','the','sarl','sas','sasu','eurl','sci','snc','gmbh','ag','bv','srl','ltda','spa','nv','oy','ab'}
UNIT_WORDS = {'unit','apt','apartment','ste','suite','pmb','room','rm','flat','fl','floor','office','shop'}
UNIT_KEEP_NUMBER = {'flat','shop','office'}      # Indian "Flat No 202" carries real information: drop the word, keep the number
PLACE_WORDS = {'city','cdp','township','twp','county','borough','of','the','near','opp','opposite','behind','beside','next','to',
 'and','at','po','post','vill','village','dist','district','taluka','tehsil','urban','rural','area','na','null','none','nil'}
FLOOR_PREFIX = {'ground', 'grd', 'basement', 'top'}
NA_COMPONENTS = {'null', 'none', 'na', 'nil', 'n a'}
DOMAIN_TLDS = {'com', 'net', 'org', 'in', 'io', 'co'}

ORDINAL_RE = re.compile(r'^\d+(st|nd|rd|th)$')
INDIC_RE = re.compile('[' + chr(0x0900) + '-' + chr(0x0DFF) + ']')   # literal chars, not 'ऀ' escapes (RE2 compatibility)
DOMAIN_RE = re.compile(r'^\s*[a-z0-9#\-]+\.?(com|net|org|in|io|co)\s*$', re.I)
URL_RE = re.compile(r'(?:https?://|www\.)\S+', re.I)
PHONE_RE = re.compile(r'\b\d{7,}\b')
ALIAS_RE = re.compile(r'^.*?\b(?:d\W?b\W?a|doing\s+business\s+as|f\W?k\W?a|formerly(?:\s+known\s+as)?|t\W?a|a\W?k\W?a|trading\s+as)\b\W*', re.I)
DOTTED_RE = re.compile(r'\b(?:[a-z]\.){2,}')          # L.L.C. -> llc, P.C. -> pc
APOS_RE = re.compile(r"['’`]")                          # removed WITHOUT a space: Orelee's -> orelees
PUNCT_RE = re.compile(r"[!-/:-@\[-`{-~¡-¿‐-‧‰-⁞　-〿]")   # -> space; keeps every script's letters
NA_RE = re.compile(r'\bn/a\b|<null>', re.I)
POBOX_RE = re.compile(r'\bp\.?\s?o\.?\s?box\b\s*[\w-]*', re.I)
HOUSE_PREFIX_RE = re.compile(r'(?:\b(?:h\.?\s?no\.?|hno|h\.?n\.?|door\s*no\.?|plot\s*no\.?|flat\s*no\.?|shop\s*no\.?|d\.?\s?no\.?|'
                             r'sy\.?\s?no\.?|kh\.?\s?no\.?|gat\s*no\.?|survey\s*no\.?|no\.?|n\s?[º°])|#+)\s*(?=\d)', re.I)
MIXED_RE = re.compile(r'(?=.*[a-z])(?=.*\d)')
LEET = str.maketrans({'0':'o','1':'l','3':'e','4':'a','5':'s','6':'g','7':'t','8':'b','9':'g','2':'z'})


def strip_latin_accents(s):
    """NFKD + drop combining marks, but only for strings whose letters are all Latin (ord < 0x250).
    Indic strings are returned untouched: NFKD would separate the vowel signs from their consonants."""
    if any(ord(ch) >= 0x250 for ch in s if ch.isalpha()):
        return s
    return ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))


_SOUNDEX = {c: d for d, letters in {'1': 'bfpv', '2': 'cgjkqsxz', '3': 'dt', '4': 'l', '5': 'mn', '6': 'r'}.items() for c in letters}


def soundex(tok):
    """Standard 4-character Soundex on the ASCII letters of `tok` ('' when it has none)."""
    letters = [c for c in tok.lower() if 'a' <= c <= 'z']
    if not letters:
        return ''
    code = letters[0].upper()
    prev = _SOUNDEX.get(letters[0], '')
    for c in letters[1:]:
        d = _SOUNDEX.get(c, '')
        if d:
            if d != prev:
                code += d
            prev = d
        elif c not in 'hw':          # vowels separate equal codes, h/w do not
            prev = ''
    return (code + '000')[:4]


def map_state(comp, country):
    """Component-level state lookup for countries that have a table ('' otherwise / for unknown countries)."""
    table = COUNTRY_STATES.get((country or '').strip().lower())
    if table is None:
        return ''
    names, codes = table
    c = comp.strip().lower()
    if len(c) == 2 and c in codes:
        return c
    return names.get(c, '')


def name_tokens(raw):
    """Raw business name -> (tokens, is_domain).  See the markdown cell above for the evidence behind each step."""
    s = raw if isinstance(raw, str) else ''
    low = s.strip().lower()
    # domain-shaped = single token like strategichoovercom / wilfordhancock.com; a bare 'franklin' or 'costco'
    # also ends in a TLD but is a normal name, so the no-dot form must end in 'com'
    is_domain = bool(DOMAIN_RE.match(s)) and ('.' in low or low.endswith('com'))
    s = s.split('|', 1)[0]                     # '| www.procopio.com' style appendix
    s = URL_RE.sub(' ', s)
    s = PHONE_RE.sub(' ', s)                   # 'LLC - 7418920210'
    s = ID_RE.sub(' ', s)                      # '(ID: 75168)'
    s = s.replace('&', ' and ')
    s = s.lower()
    s = strip_latin_accents(s)
    m = HONORIFIC_RE.match(s)                  # S2/S3 prepend 'Smt' / 'Shri' / 'Sri' / 'Dr' / 'M/s' to Indian names
    if m and any(ch.isalpha() for ch in s[m.end():]):
        s = s[m.end():]
    m = ALIAS_RE.match(s)                      # 'X DBA Y' / 'X doing business as Y' / 'X f/k/a Y': keep Y
    if m:
        rest = s[m.end():]
        if any(ch.isalpha() for ch in rest):
            s = rest
    s = DOTTED_RE.sub(lambda mm: mm.group(0).replace('.', ''), s)
    s = APOS_RE.sub('', s)
    s = PUNCT_RE.sub(' ', s)
    toks = s.split()
    if is_domain:
        if len(toks) >= 2 and toks[-1] in DOMAIN_TLDS:
            toks = toks[:-1]
        elif len(toks) == 1 and toks[0].endswith('com') and len(toks[0]) > 6:
            toks = [toks[0][:-3]]
    return toks, is_domain


def clean_name(raw, translit_map=None):
    """-> (name_norm, name_key, leet_key, is_indic, is_domain)."""
    toks, is_domain = name_tokens(raw)
    is_indic = any(INDIC_RE.search(t) for t in toks)
    if is_indic:
        if translit_map:
            toks = [translit_map.get(t, t) for t in toks]
        if HAVE_UNIDECODE:
            out = []
            for t in toks:
                if INDIC_RE.search(t):
                    t = PUNCT_RE.sub('', unidecode(t).lower().replace(' ', ''))
                if t:
                    out.append(t)
            toks = out
    ded = []                                   # 'Creative Wireless Wireless Networks' -> repeated words
    for t in toks:
        if not ded or ded[-1] != t:
            ded.append(t)
    core = [t for t in ded if t not in LEGAL_TOKENS] or ded
    leet = [t.translate(LEET) if (MIXED_RE.match(t) and not ORDINAL_RE.match(t)) else t for t in core]
    name_norm = ' '.join(core)                                    # ORDERED: string similarity features
    name_key = ' '.join(sorted(set(core)))                        # order-free: blocking / equality
    leet_key = ' '.join(sorted(set(leet))) if leet != core else ''   # C0rrine -> corrine, kept as a SECOND key
    return name_norm, name_key, leet_key, int(is_indic), int(is_domain)


def clean_address(raw, country):
    """-> (addr_norm, addr_key, state, geo, city, house, street, postcode, has_addr)."""
    if not isinstance(raw, str) or not raw.strip():
        return ('', '', '', '', '', '', '', '', 0)
    s = raw.lower()
    s = strip_latin_accents(s)
    s = s.replace('&', ' and ')
    s = NA_RE.sub(' ', s)
    s = POBOX_RE.sub(' ', s)                   # inserted 'PO Box 123' is noise, not location
    comps = [c.strip() for c in s.split(',')]
    comps = [c for c in comps if c and c not in NA_COMPONENTS]
    # STATE: component level only ('Washington Avenue' must never become a state); the last mapped
    # component wins but an explicit 2-letter code beats a spelled-out name
    state, state_idx, state_is_code = '', -1, False
    for i, c in enumerate(comps):
        m = map_state(c, country)
        if m:
            is_code = len(c.strip()) == 2
            if is_code or not state_is_code:
                state, state_idx, state_is_code = m, i, is_code
    if state_idx >= 0:
        comps = comps[:state_idx] + comps[state_idx + 1:]
    kept_all, comp_tokens = [], []
    postcode = house = street = ''
    for c in comps:
        c2 = HOUSE_PREFIX_RE.sub(' ', c)      # 'H.No 12', 'Plot No 7', '##700' -> bare number
        c2 = PUNCT_RE.sub(' ', c2)
        raw_toks = c2.split()
        # POSTCODE: a 5-6 digit token in a component with <= 2 tokens ('75002 paris', standalone '69006',
        # Indian PIN) - never inside '34184 tanglewood pass' (5-digit US house numbers); a 6-digit token that
        # ends a component ('new delhi 110001') is a PIN too.  Kept verbatim (leading zeros are significant).
        pc_here = ''
        if not postcode:
            for j, t in enumerate(raw_toks):
                if t.isdigit() and ((len(raw_toks) <= 2 and 5 <= len(t) <= 6) or (len(t) == 6 and j == len(raw_toks) - 1)):
                    pc_here = t
                    break
        toks = []
        skip_next = False
        for t in raw_toks:
            if skip_next:
                skip_next = False
                if (len(t) <= 4 and any(ch.isdigit() for ch in t)) or len(t) == 1:
                    continue
            if t == pc_here:
                continue
            t = ABBR.get(t, t)
            t = CITY_SYN.get(t, t)
            if t.isdigit():
                t = t.lstrip('0') or '0'       # '05205' -> '5205'
            if t in UNIT_WORDS:
                if t == 'fl' and toks and (ORDINAL_RE.match(toks[-1]) or toks[-1] in FLOOR_PREFIX):
                    toks.pop()                 # '2nd Fl', 'ground floor'
                skip_next = t not in UNIT_KEEP_NUMBER
                continue
            if t in PLACE_WORDS:               # 'Seattle CITY', '... CDP', '... County', 'NULL'
                continue
            toks.append(t)
        if pc_here:
            postcode = pc_here
        if not house:
            for j, t in enumerate(toks):
                if t.isdigit() and len(t) <= 6 or (len(t) <= 7 and t[:-1].isdigit() and t[-1].isalpha()):
                    house = t if t.isdigit() else t[:-1]      # '1098d' -> '1098'
                    for t2 in toks[j + 1:]:
                        if ((len(t2) >= 3 and t2.isalpha()) or ORDINAL_RE.match(t2)) and t2 not in UNIT_WORDS \
                                and t2 not in PLACE_WORDS and t2 not in STREET_SKIP:
                            street = t2
                            break
                    break
        comp_tokens.append(toks)
        kept_all.extend(toks)
    city = ''
    for toks in reversed(comp_tokens):        # last component without digits = the city
        if toks and not any(any(ch.isdigit() for ch in t) for t in toks):
            city = ' '.join(toks)
            break
    if not city and comp_tokens:
        city = ' '.join(t for t in comp_tokens[-1] if t.isalpha())
    if state:
        kept_all.append(state)
    addr_norm = ' '.join(kept_all)
    addr_key = ' '.join(sorted(set(kept_all)))
    # open-set blocking geography: state -> postcode prefix -> last words of the last TWO digit-free components.
    # French addresses have no postcode and end with a region OR a departement that differs across sources
    # ('Bordeaux, Nouvelle-Aquitaine' vs 'MERIGNAC, Gironde'), and Source-1 shuffles components, so both the
    # city and the region word are emitted as blocking geographies (space-separated; keys are built per word).
    if state:
        geo = state
    elif postcode:
        geo = postcode[:2]
    else:
        nodigit = [toks for toks in comp_tokens if toks and not any(any(ch.isdigit() for ch in t) for t in toks)]
        geo = ' '.join(dict.fromkeys(toks[-1] for toks in nodigit[-2:]))
    return (addr_norm, addr_key, state, geo, city, house, street, postcode, int(bool(addr_norm or postcode)))


# ---- sanity checks (the marked ones are asserted) ---------------------------------------------------
print("=== name normalization ===")
_name_cases = [
    ('Avizephdrex DBA LUK Energetics Inc', dict(name_norm='luk energetics', name_key='energetics luk'), True),
    ('[Llc] Optimal Superior Hudson,', dict(name_norm='optimal superior hudson'), False),
    ('Procopio, Florence and Watford Services | [www.procopio.com](https://www.procopio.com)', dict(name_norm='procopio florence and watford services'), False),
    ('Pacific Program LLC - 7418920210', dict(name_norm='pacific program'), False),
    ('strategichoovercom', dict(name_norm='strategichoover', is_domain=1), True),
    ('C0rrine Ebanks Secure [Mountain]', dict(name_key='c0rrine ebanks mountain secure', leet_key='corrine ebanks mountain secure'), True),
    ('NEW DELHI PLTICS HOLDINGS HOLDINGS LIMITED', dict(name_norm='new delhi pltics holdings'), False),
    ('Houston & Partners Private Limited', dict(name_norm='houston and partners'), False),
    ('Boulangerie Martin SARL', dict(name_norm='boulangerie martin'), False),
]
for raw, expect, must in _name_cases:
    nn, nk, lk, ii, idm = clean_name(raw)
    got = dict(name_norm=nn, name_key=nk, leet_key=lk, is_indic=ii, is_domain=idm)
    ok = all(got[k] == v for k, v in expect.items())
    print(f"{'ASSERT ' if must else '       '}{'OK ' if ok else 'BAD'} | {raw!r:80s} -> {got}")
    if must:
        assert ok, (raw, expect, got)
print("\n=== address normalization ===")
_addr_cases = [
    (('OH, Columbus, 5559 Orville Avenue', 'US'), dict(state='oh', city='columbus', house='5559', street='orville', addr_key='5559 ave columbus oh orville'), True),
    (('05205 FERNWAY ROAD, KINGSTON, NY', 'US'), dict(house='5205', street='fernway'), True),
    (('1722. Court Saint, Siouux CITY CDP, Iowa', 'US'), dict(state='ia', house='1722', city='siouux'), False),
    (('2681 Dogwood Ridge Road, Wheelersburg, Unit C, OH', 'US'), dict(street='dogwood', state='oh', city='wheelersburg'), False),
    (('Fl-A/204 Shreebalaji Krupa Plot-19A Sec-20 Khargha, Navi Mumbai, MH', 'India'), dict(state='mh', city='navi mumbai'), False),
    (('12 Rue de la Paix, 75002 Paris', 'France'), dict(state='', postcode='75002', geo='75', city='paris', house='12'), True),
]
for (raw, ctry), expect, must in _addr_cases:
    an, ak, st, geo, city, house, street, pc, has = clean_address(raw, ctry)
    got = dict(addr_norm=an, addr_key=ak, state=st, geo=geo, city=city, house=house, street=street, postcode=pc, has_addr=has)
    ok = all(got[k] == v for k, v in expect.items())
    print(f"{'ASSERT ' if must else '       '}{'OK ' if ok else 'BAD'} | {raw!r} ({ctry}) -> {got}")
    if must:
        assert ok, (raw, expect, got)
_toks = clean_address('05205 FERNWAY ROAD, KINGSTON, NY', 'US')[0].split()
assert '5205' in _toks and 'rd' in _toks and '05205' not in _toks
_toks = clean_address('1722. Court Saint, Siouux CITY CDP, Iowa', 'US')[0].split()
assert 'ct' in _toks and 'st' in _toks, _toks
_an = clean_address('2681 Dogwood Ridge Road, Wheelersburg, Unit C, OH', 'US')[0]
assert 'unit' not in _an.split() and ' c ' not in f' {_an} ', _an
print("\nall sanity assertions passed")

## 4. Parallel normalization and compact storage

`normalize_table` runs `_normalize_rows` (a top-level function, so `fork` pickling works) in a `multiprocessing.Pool(N_JOBS)` over chunks of about 1% of the rows; each finished chunk advances the progress tracker. The results are stored column-wise in a `NormTable`: string columns as `string[pyarrow]` Series (object dtype when pyarrow is unavailable) and the three flags as `int8` arrays. Per-record dicts are never created — 12.5M of them would not fit in memory. `get(col, rows)` returns Python lists for the requested row positions only.

In [ ]:
# =====================================================================================
# Cell 6 - parallel normalization + compact storage (NormTable)
# =====================================================================================
NORM_COLS = ['name_norm', 'name_key', 'leet_key', 'is_indic', 'is_domain', 'addr_norm', 'addr_key', 'state', 'geo',
             'city', 'house', 'street', 'postcode', 'has_addr']
STR_COLS = [c for c in NORM_COLS if c not in ('is_indic', 'is_domain', 'has_addr')]
FLAG_COLS = ['is_indic', 'is_domain', 'has_addr']


def _normalize_rows(job):
    """TOP-LEVEL worker (fork pickling): (rows, translit_map) -> list of 14-tuples in NORM_COLS order."""
    rows, translit_map = job
    out = []
    for name, addr, country in rows:
        nn, nk, lk, ii, idm = clean_name(name, translit_map)
        an, ak, st, geo, city, house, street, pc, has = clean_address(addr, country)
        out.append((nn, nk, lk, ii, idm, an, ak, st, geo, city, house, street, pc, has))
    return out


def _to_str_series(values):
    """Compact string storage: string[pyarrow] when pyarrow is available, object otherwise."""
    if isinstance(values, pd.Series):                       # already a string Series: share, do not copy
        return values.reset_index(drop=True)
    if HAVE_PA:
        arr = values if isinstance(values, (pa.Array, pa.ChunkedArray)) else pa.array(values, type=pa.string())
        try:
            return pd.Series(pd.arrays.ArrowStringArray(arr if isinstance(arr, pa.ChunkedArray) else pa.chunked_array([arr])))
        except Exception:
            return pd.Series(arr.to_pylist(), dtype='string[pyarrow]')
    return pd.Series(list(values) if not isinstance(values, list) else values, dtype=object)


class NormTable:
    """Column store for one normalized source (12.5M per-record dicts would not fit in memory)."""
    def __init__(self, df, columns, flags, label=''):
        self.n = len(df)
        self.label = label
        self.cols = {}
        self.cols['entity_id'] = _to_str_series(df['entity_id'])
        codes, uniques = pd.factorize(df['country'])                     # few distinct values: map them, not rows
        canon = np.asarray([canon_country(u) for u in uniques] + [''], dtype=object)
        self.cols['country'] = _to_str_series(canon[np.where(codes < 0, len(uniques), codes)].tolist())
        self.cols['raw_name'] = _to_str_series(df['business_name'])
        self.cols['raw_addr'] = _to_str_series(df['business_address'])   # NA stays NA here, '' in get()
        for c in STR_COLS:
            self.cols[c] = _to_str_series(columns[c])
        self.flags = {c: np.asarray(flags[c], dtype=np.int8) for c in FLAG_COLS}

    def get(self, col, idx=None):
        """Python list (numpy int8 array for flags) of the requested row positions (all rows when idx is None)."""
        if col in self.flags:
            a = self.flags[col]
            return a if idx is None else a[idx]
        s = self.cols[col]
        out = s.tolist() if idx is None else s.iloc[np.asarray(idx)].tolist()
        if col == 'raw_addr':
            return [x if isinstance(x, str) else '' for x in out]
        return out

    def __len__(self):
        return self.n


def normalize_table(df, translit_map, label):
    """Normalize every row of df with multiprocessing.Pool(N_JOBS); each finished chunk advances the Progress
    tracker by ~1%.  Returns a NormTable."""
    t0 = time.time()
    n = len(df)
    chunk = max(10_000, n // 100)
    names, addrs, countries = df['business_name'], df['business_address'], df['country']

    def jobs():                                   # one chunk of Python strings at a time, never all 12.5M rows
        for i in range(0, n, chunk):
            yield (list(zip(names.iloc[i:i + chunk].tolist(), addrs.iloc[i:i + chunk].tolist(),
                            countries.iloc[i:i + chunk].tolist())), translit_map)
    n_jobs = (n + chunk - 1) // chunk
    prog = progress_for(n_jobs, f"normalize {label}")
    str_parts = {c: [] for c in STR_COLS}
    flag_parts = {c: [] for c in FLAG_COLS}

    def consume(res):
        cols = list(zip(*res)) if res else [[] for _ in NORM_COLS]
        for k, c in enumerate(NORM_COLS):
            if c in FLAG_COLS:
                flag_parts[c].append(np.asarray(cols[k], dtype=np.int8))
            else:
                str_parts[c].append(pa.array(list(cols[k]), type=pa.string()) if HAVE_PA else list(cols[k]))
        prog.update(1)

    for res in pool_imap(_normalize_rows, jobs(), N_JOBS if n_jobs > 1 else 1, f"normalize {label}"):
        consume(res)
    prog.close()
    columns = {}
    for c in STR_COLS:
        if HAVE_PA:
            columns[c] = pa.chunked_array(str_parts[c]) if str_parts[c] else pa.chunked_array([pa.array([], type=pa.string())])
        else:
            columns[c] = [x for part in str_parts[c] for x in part]
    flags = {c: (np.concatenate(flag_parts[c]) if flag_parts[c] else np.zeros(0, np.int8)) for c in FLAG_COLS}
    nt = NormTable(df, columns, flags, label)
    del str_parts, flag_parts, columns
    gc.collect()
    log(f"normalized {label}: {n:,} rows in {time.time() - t0:.1f}s | Indic names {int(nt.flags['is_indic'].sum()):,} | "
        f"domain-style {int(nt.flags['is_domain'].sum()):,} | no address {int((nt.flags['has_addr'] == 0).sum()):,}")
    return nt

## 5. Transliteration dictionary (training split only)

Indic-script pool names are word-by-word transliterations of the English Source-1 name (`शिव वेंचर्स प्राइवेट लिमिटेड` ↔ `Shiv Ventures Private Limited`); `unidecode` alone scores below 80 on 98% of such true pairs. For the **training** entities only, the tokens of every (Source-1 name, Indic pool name) true pair are aligned positionally when the token counts agree, and each Indic token keeps its most frequent Latin target when it was seen at least twice with a share of at least 50%. The dictionary is saved with the artifacts and applied unchanged to the validation and test data; unmapped Indic tokens fall back to `unidecode` when it is installed and are otherwise left as they are.

In [ ]:
# =====================================================================================
# Cell 7 - transliteration dictionary learned from the TRAINING split only
# =====================================================================================
def learn_translit_map(s1_df, pool_df, gt, train_s1_ids, min_count=2, min_share=0.5, s1_pos=None, pool_pos=None):
    """Indic-script pool names are word-by-word transliterations of the English Source-1 name
    (unidecode alone scores < 80 on 98% of such true pairs).  For the TRAINING entities only, align the
    tokens of every (S1 name, Indic pool name) true pair positionally when the token counts agree and keep,
    for each Indic token, its most frequent Latin target (count >= min_count and share >= min_share).
    The dictionary is stored in the artifacts and applied unchanged at inference."""
    t0 = time.time()
    s1_pos = s1_pos if s1_pos is not None else pd.Index(s1_df['entity_id'])
    pool_pos = pool_pos if pool_pos is not None else pd.Index(pool_df['entity_id'])
    s1_list, p_list = [], []
    for s1_id in train_s1_ids:
        for pid in gt.get(s1_id, ()):
            s1_list.append(s1_id)
            p_list.append(pid)
    si = s1_pos.get_indexer(s1_list)
    pi = pool_pos.get_indexer(p_list)
    ok = (si >= 0) & (pi >= 0)
    s1_names = s1_df['business_name'].iloc[si[ok]].tolist()
    pool_names = pool_df['business_name'].iloc[pi[ok]].tolist()
    counts = defaultdict(Counter)
    n_indic = n_aligned = 0
    for a, b in zip(s1_names, pool_names):
        if not isinstance(b, str) or not INDIC_RE.search(b):
            continue
        n_indic += 1
        ta, _ = name_tokens(a)
        tb, _ = name_tokens(b)
        if not ta or len(ta) != len(tb):
            continue
        n_aligned += 1
        for x, y in zip(tb, ta):
            if INDIC_RE.search(x) and not INDIC_RE.search(y):
                counts[x][y] += 1
    tmap = {}
    for x, c in counts.items():
        y, n = c.most_common(1)[0]
        if n >= min_count and n / sum(c.values()) >= min_share:
            tmap[x] = y
    log(f"transliteration map: {len(tmap):,} Indic tokens learned from {n_indic:,} Indic training pairs "
        f"({n_aligned:,} positionally aligned) in {time.time() - t0:.1f}s")
    return tmap

## 6. Blocking

Every key string is prefixed with the (canonicalised) country so that partitions never cross countries. Keys are hashed with `pd.util.hash_array` (vectorised, stable across processes) and stored per family as a sorted `uint64` array with parallel `int32` row positions; blocks larger than the family's cap are dropped (generic names such as `Primary Care` × 741 would otherwise flood the candidate lists). A query hashes its keys, finds the block ranges with `searchsorted`, expands them vectorially, unions the (query, record) pairs through a 64-bit code and OR-accumulates a per-family hit mask. Candidates are ranked by `nhits + 3·hit_n + 3·hit_a + 2·hit_nc` and cut at `MAX_CAND_PER_S1 = 50` per Source-1 entity. **This capped set is what the model scores and exactly what `candidate_pairs.tsv` contains.**

| family | built from | catches |
|---|---|---|
| `n` | country + `name_key` (and `leet_key`) | exact name after cleaning: caps, legal tokens, word order, brackets, junk, digit typos |
| `nc` | country + geo + `name_key` / `leet_key` (query side also geo-less) | the same within one state / region, reaching address-less pool rows through the geo-less variant |
| `a` | country + `addr_key` | identical address after cleaning even when the name is unrecognisable (aliases, domain-style names) |
| `g1` | country + geo + first token (≥ 3 chars) | typos and edits in later tokens |
| `hs` | country + geo + house number + street name | matching street address with a corrupted name |
| `p5` | country + geo + first 5 characters of the space-free name (query side also for tokens 2–5) | domain-style concatenations, transpositions and deletions after the prefix |
| `ph` | country + geo + Soundex of the first two tokens | phonetic / typo variants of the leading words |
| `tk` | country + geo + each token (≥ 3 chars) | any shared distinctive token (word-order changes, inserted or dropped words) |
| `tk0` | country + token, index side only for address-less records | address-less pool rows with a shared token |

In [ ]:
# =====================================================================================
# Cell 8 - blocking: hashed sorted-array index over nine key families
# =====================================================================================
KEY_TYPES = ['n', 'nc', 'a', 'g1', 'hs', 'p5', 'ph', 'tk', 'tk0']
KEY_BIT = {k: 1 << i for i, k in enumerate(KEY_TYPES)}
BIT_N, BIT_A, BIT_NC = KEY_BIT['n'], KEY_BIT['a'], KEY_BIT['nc']


def hash_keys(strs):
    """Vectorized, process-stable uint64 hashes (pd.util.hash_array uses a fixed key; Python's hash() is
    salted per process and must never be used across Pool workers)."""
    if len(strs) == 0:
        return np.zeros(0, dtype=np.uint64)
    return pd.util.hash_array(np.asarray(strs, dtype=object))


def _gen_keys_from_columns(cols, rows, query):
    """cols: dict of Python lists aligned with `rows` (absolute row positions).  -> {type: (keys, rows)}"""
    C, G, NK, LK, NN = cols['country'], cols['geo'], cols['name_key'], cols['leet_key'], cols['name_norm']
    AK, HA, H, S = cols['addr_key'], cols['has_addr'], cols['house'], cols['street']
    out = {k: ([], []) for k in KEY_TYPES}
    kn, rn = out['n']; knc, rnc = out['nc']; ka, ra = out['a']; kg1, rg1 = out['g1']; khs, rhs = out['hs']
    kp5, rp5 = out['p5']; kph, rph = out['ph']; ktk, rtk = out['tk']; ktk0, rtk0 = out['tk0']
    for i in range(len(rows)):
        nk = NK[i]
        if not nk:
            continue
        r = int(rows[i]); c = C[i]; g = G[i]; lk = LK[i]; toks = NN[i].split()
        G_ = g.split() or ['']                        # up to two geo words for open-set countries
        GQ = G_ + [''] if (query and g) else G_      # query side also emits geo-less variants (address-less pool rows)
        kn.append(f"{c}|{nk}"); rn.append(r)
        if lk:
            kn.append(f"{c}|{lk}"); rn.append(r)
        for gg in GQ:
            knc.append(f"{c}|{gg}|{nk}"); rnc.append(r)
            if lk:
                knc.append(f"{c}|{gg}|{lk}"); rnc.append(r)
        if HA[i] and AK[i]:
            ka.append(f"{c}|{AK[i]}"); ra.append(r)
        if not toks:
            continue
        t0 = toks[0]
        if len(t0) >= 3:
            for gg in G_:
                kg1.append(f"{c}|{gg}|{t0}"); rg1.append(r)
        if H[i] and S[i]:
            for gg in G_:
                khs.append(f"{c}|{gg}|{H[i]}|{S[i]}"); rhs.append(r)
        nospace = ''.join(toks)
        sx = soundex(t0) + (soundex(toks[1]) if len(toks) > 1 else '')
        for gg in GQ:
            if len(nospace) >= 5:
                kp5.append(f"{c}|{gg}|{nospace[:5]}"); rp5.append(r)
            if query:                                 # domain-style concatenations start with any token
                for t in toks[1:5]:
                    if len(t) >= 4:
                        kp5.append(f"{c}|{gg}|{t[:5]}"); rp5.append(r)
            kph.append(f"{c}|{gg}|{sx}"); rph.append(r)
        seen = set()
        for t in toks:
            if len(t) >= 3 and t not in seen:
                seen.add(t)
                for gg in G_:
                    ktk.append(f"{c}|{gg}|{t}"); rtk.append(r)
                if query or not HA[i]:
                    ktk0.append(f"{c}|{t}"); rtk0.append(r)
    return out


KEY_COLS = ['country', 'geo', 'name_key', 'leet_key', 'name_norm', 'addr_key', 'has_addr', 'house', 'street']


def generate_keys(nt, rows, query):
    """{key_type: (key_strings, row_positions)} for the records at `rows` of NormTable `nt`."""
    rows = np.asarray(rows)
    cols = {c: (nt.get(c, rows).tolist() if c == 'has_addr' else nt.get(c, rows)) for c in KEY_COLS}
    return _gen_keys_from_columns(cols, rows, query)


def _block_keys_job(job):
    """TOP-LEVEL Pool worker: (cols, rows) -> {type: (hashes, rows)} plus the hashed name-frequency keys."""
    cols, rows = job
    keys = _gen_keys_from_columns(cols, rows, query=False)
    out = {k: (hash_keys(v[0]), np.asarray(v[1], dtype=np.int32)) for k, v in keys.items() if v[0]}
    nf = [f"{c}|{nk}" for c, nk in zip(cols['country'], cols['name_key']) if nk]
    out['__nf__'] = hash_keys(nf)
    return out


def popcount16(hits):
    return np.unpackbits(np.ascontiguousarray(hits).view(np.uint8)).reshape(-1, 16).sum(axis=1).astype(np.int8)


class KeyCounter:
    """Frequency table of hashed key strings (sorted unique hashes + counts)."""
    def __init__(self, hashes):
        hashes = np.asarray(hashes, dtype=np.uint64)
        self.hashes, self.counts = np.unique(hashes, return_counts=True) if len(hashes) else (np.zeros(0, np.uint64), np.zeros(0, np.int64))
        self.series = pd.Series(self.counts, index=self.hashes)
    @classmethod
    def from_keys(cls, key_strs):
        return cls(hash_keys([k for k in key_strs if k]))
    def lookup(self, key_strs):
        if len(key_strs) == 0 or len(self.hashes) == 0:
            return np.zeros(len(key_strs), dtype=np.int64)
        h = hash_keys(list(key_strs))
        i = np.minimum(np.searchsorted(self.hashes, h), len(self.hashes) - 1)
        return np.where(self.hashes[i] == h, self.counts[i], 0)


class BlockIndex:
    """Per key family: (sorted uint64 hashes, int32 row positions); blocks larger than caps[type] are dropped."""
    def __init__(self, nt, caps, label='pool'):
        t0 = time.time()
        self.caps = dict(caps)
        n = len(nt)
        chunk = 500_000
        starts = list(range(0, n, chunk))
        prog = progress_for(len(starts), f"block index {label}")
        parts = {k: ([], []) for k in KEY_TYPES}
        nf_parts = []
        self.dropped = {k: 0 for k in KEY_TYPES}

        def jobs():
            for s in starts:
                rows = np.arange(s, min(n, s + chunk), dtype=np.int32)
                cols = {c: (nt.get(c, rows).tolist() if c == 'has_addr' else nt.get(c, rows)) for c in KEY_COLS}
                yield (cols, rows)

        def consume(res):
            for k in KEY_TYPES:
                if k in res:
                    parts[k][0].append(res[k][0]); parts[k][1].append(res[k][1])
            nf_parts.append(res['__nf__'])
            prog.update(1)

        for res in pool_imap(_block_keys_job, jobs(), N_JOBS if len(starts) > 1 else 1, f"block index {label}"):
            consume(res)
        prog.close()
        self.index = {}
        stats = []
        for k in KEY_TYPES:
            if not parts[k][0]:
                continue
            h = np.concatenate(parts[k][0]); r = np.concatenate(parts[k][1])
            order = np.argsort(h, kind='stable')
            h = h[order]; r = r[order]
            uniq, first, counts = np.unique(h, return_index=True, return_counts=True)
            big = counts > self.caps.get(k, 10 ** 9)
            self.dropped[k] = int(big.sum())
            if big.any():
                keep = np.repeat(~big, counts)            # unique() is sorted, so counts align with sorted h
                h = h[keep]; r = r[keep]
            self.index[k] = (h, r)
            stats.append(f"{k}: {len(h):,} entries / {len(uniq) - self.dropped[k]:,} blocks (dropped {self.dropped[k]:,} > {self.caps.get(k)})")
            parts[k] = None
        self.name_freq = KeyCounter(np.concatenate(nf_parts) if nf_parts else np.zeros(0, np.uint64))
        del parts, nf_parts
        gc.collect()
        log(f"block index {label}: {n:,} records in {fmt_secs(time.time() - t0)} | " + ' | '.join(stats))

    def query(self, nt_q, rows, max_per_s1):
        """Candidates for the Source-1 rows `rows` -> DataFrame(q int32, p int32, hits int16, nhits int8),
        sorted by q, at most max_per_s1 per q.  This capped set is what the model scores AND what
        candidate_pairs.tsv contains."""
        keys = generate_keys(nt_q, rows, query=True)
        qs, ps, bs = [], [], []
        for k in KEY_TYPES:
            strs, qrows = keys[k]
            if not strs or k not in self.index:
                continue
            h = hash_keys(strs)
            sh, srows = self.index[k]
            left = np.searchsorted(sh, h, side='left')
            right = np.searchsorted(sh, h, side='right')
            sizes = right - left
            total = int(sizes.sum())
            if total == 0:
                continue
            offsets = np.arange(total) - np.repeat(np.cumsum(sizes) - sizes, sizes)
            pos = np.repeat(left, sizes) + offsets
            ps.append(srows[pos])
            qs.append(np.repeat(np.asarray(qrows, dtype=np.int32), sizes))
            bs.append(np.full(total, KEY_BIT[k], dtype=np.int16))
        empty = pd.DataFrame({'q': np.zeros(0, np.int32), 'p': np.zeros(0, np.int32),
                              'hits': np.zeros(0, np.int16), 'nhits': np.zeros(0, np.int8)})
        if not qs:
            return empty
        q = np.concatenate(qs); p = np.concatenate(ps); b = np.concatenate(bs)
        code = (q.astype(np.int64) << 32) | p.astype(np.int64)
        uniq, inv = np.unique(code, return_inverse=True)
        hits = np.zeros(len(uniq), dtype=np.int16)
        np.bitwise_or.at(hits, inv, b)
        q = (uniq >> 32).astype(np.int32)
        p = (uniq & 0xFFFFFFFF).astype(np.int32)
        nhits = popcount16(hits)
        score = (nhits.astype(np.int32) + 3 * ((hits & BIT_N) > 0) + 3 * ((hits & BIT_A) > 0) + 2 * ((hits & BIT_NC) > 0))
        order = np.lexsort((p, -score, q))
        q, p, hits, nhits = q[order], p[order], hits[order], nhits[order]
        starts = np.r_[0, np.flatnonzero(np.diff(q)) + 1]
        counts = np.diff(np.r_[starts, len(q)])
        rank = np.arange(len(q)) - np.repeat(starts, counts)
        keep = rank < max_per_s1
        return pd.DataFrame({'q': q[keep], 'p': p[keep], 'hits': hits[keep], 'nhits': nhits[keep]})

## 7. Pair features (exactly 77, float32, fixed order)

For every candidate pair the notebook computes: equality of the cleaned names / keys / leet keys (1–3); RapidFuzz similarities on the ordered names — ratio, partial ratio, token-set ratio, token-sort ratio, WRatio, Jaro-Winkler and normalised Levenshtein (4–10) — and on the space-free names (11–13); token-set statistics and an IDF-weighted overlap (14–18); character (2–4-gram) and word TF-IDF cosines (19–20); token counts, lengths and their ratio (21–25); first-token and Soundex agreement (26–27); Indic / domain flags (28–30); address agreement — missing flag, exact key equality, state / geo / city / house / street / postcode agreement with Jaro-Winkler and Levenshtein similarities, token-set statistics, token-set ratio, TF-IDF cosine and numeric-token Jaccard (31–50); the candidate's source (51); the log frequency of the Source-1 name key in the pool and in Source-1 (52–53: generic names are never a decision on their own); the nine blocking hits and their count (54–63); a pre-score (64); the log number of candidates (65) and, for `n_tsr`, `n_tfidf_char`, `pre_score` and `a_jacc`, the gap to the entity's best candidate, the descending rank and the entity maximum (66–77). Rank / gap features need the entity's complete candidate list, so pair chunks are cut only at entity boundaries. The TF-IDF vocabularies and IDF weights are fitted on unsupervised text of the training split only. Country is never a feature.

In [ ]:
# =====================================================================================
# Cell 9 - pair features (exactly 77, float32, fixed order)
# =====================================================================================
if HAVE_RF:
    SC = {'ratio': rf_fuzz.ratio, 'partial': rf_fuzz.partial_ratio, 'tsr': rf_fuzz.token_set_ratio,
          'tsort': rf_fuzz.token_sort_ratio, 'wr': rf_fuzz.WRatio, 'jw': rf_JW.normalized_similarity,
          'levsim': rf_Lev.normalized_similarity}

    def rf_batch(scorer, a, b):
        """Element-wise scorer over two aligned lists (RapidFuzz C++ threads; per-pair loop otherwise)."""
        if len(a) == 0:
            return np.zeros(0, dtype=np.float32)
        if hasattr(rf_process, 'cpdist'):
            return np.asarray(rf_process.cpdist(a, b, scorer=scorer, workers=N_JOBS), dtype=np.float32)
        return np.fromiter((scorer(x, y) for x, y in zip(a, b)), dtype=np.float32, count=len(a))
else:
    import difflib

    def _py_ratio(a, b):
        if not a and not b:
            return 100.0
        if not a or not b:
            return 0.0
        return 100.0 * difflib.SequenceMatcher(None, a, b).ratio()

    def _py_partial(a, b):
        if not a or not b:
            return 100.0 if a == b else 0.0
        s, l = (a, b) if len(a) <= len(b) else (b, a)
        best = 0.0
        for blk in difflib.SequenceMatcher(None, s, l).get_matching_blocks():
            start = max(0, min(blk.b - blk.a, len(l) - len(s)))
            best = max(best, _py_ratio(s, l[start:start + len(s)]))
            if best >= 100.0:
                break
        return best

    def _py_tsort(a, b):
        return _py_ratio(' '.join(sorted(a.split())), ' '.join(sorted(b.split())))

    def _py_tsr(a, b):
        sa, sb = set(a.split()), set(b.split())
        inter = ' '.join(sorted(sa & sb))
        d1 = ' '.join(sorted(sa - sb)); d2 = ' '.join(sorted(sb - sa))
        t1 = (inter + ' ' + d1).strip(); t2 = (inter + ' ' + d2).strip()
        return max(_py_ratio(inter, t1), _py_ratio(inter, t2), _py_ratio(t1, t2))

    def _py_wratio(a, b):
        if not a or not b:
            return 0.0
        base = _py_ratio(a, b)
        lr = max(len(a), len(b)) / max(1, min(len(a), len(b)))
        if lr < 1.5:
            return max(base, 0.95 * _py_tsort(a, b), 0.95 * _py_tsr(a, b))
        scale = 0.9 if lr < 8 else 0.6
        return max(base, scale * _py_partial(a, b), 0.95 * scale * _py_tsort(a, b), 0.95 * scale * _py_tsr(a, b))

    def _py_jw(a, b):
        if not a or not b:
            return 1.0 if a == b else 0.0
        la, lb = len(a), len(b)
        rng = max(la, lb) // 2 - 1
        ma = [False] * la; mb = [False] * lb
        m = 0
        for i in range(la):
            lo, hi = max(0, i - rng), min(lb, i + rng + 1)
            for j in range(lo, hi):
                if not mb[j] and a[i] == b[j]:
                    ma[i] = mb[j] = True; m += 1
                    break
        if m == 0:
            return 0.0
        t = 0; k = 0
        for i in range(la):
            if ma[i]:
                while not mb[k]:
                    k += 1
                if a[i] != b[k]:
                    t += 1
                k += 1
        jaro = (m / la + m / lb + (m - t / 2) / m) / 3.0
        pre = 0
        for i in range(min(4, la, lb)):
            if a[i] == b[i]:
                pre += 1
            else:
                break
        return jaro + 0.1 * pre * (1 - jaro)

    def _py_levsim(a, b):
        if not a and not b:
            return 1.0
        if not a or not b:
            return 0.0
        prev = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            cur = [i]
            for j, cb in enumerate(b, 1):
                cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
            prev = cur
        return 1.0 - prev[-1] / max(len(a), len(b))

    SC = {'ratio': _py_ratio, 'partial': _py_partial, 'tsr': _py_tsr, 'tsort': _py_tsort, 'wr': _py_wratio,
          'jw': _py_jw, 'levsim': _py_levsim}

    def rf_batch(scorer, a, b):
        return np.fromiter((scorer(x, y) for x, y in zip(a, b)), dtype=np.float32, count=len(a))


def tfidf_cosine(vec, a, b):
    """Row-wise cosine between the TF-IDF vectors of aligned string lists; only the unique strings of the
    chunk are transformed (np.unique on object arrays); the vectorizer output is l2-normalised."""
    n = len(a)
    if n == 0:
        return np.zeros(0, dtype=np.float32)
    allv = np.concatenate([np.asarray(a, dtype=object), np.asarray(b, dtype=object)])
    uniq, inv = np.unique(allv, return_inverse=True)
    X = vec.transform(uniq.tolist())
    Xa = X[inv[:n]]; Xb = X[inv[n:]]
    return np.asarray(Xa.multiply(Xb).sum(axis=1)).ravel().astype(np.float32)


def set_stats(a_keys, b_keys):
    """Jaccard, containment in a, containment in b, #common over whitespace-split token sets."""
    n = len(a_keys)
    jac = np.zeros(n, np.float32); ca = np.zeros(n, np.float32); cb = np.zeros(n, np.float32); com = np.zeros(n, np.float32)
    for i in range(n):
        A = a_keys[i]; B = b_keys[i]
        if not A or not B:
            continue
        sa = set(A.split()); sb = set(B.split())
        c = len(sa & sb)
        if c:
            jac[i] = c / len(sa | sb); ca[i] = c / len(sa); cb[i] = c / len(sb); com[i] = c
    return jac, ca, cb, com


def idf_overlap(a_keys, b_keys, idf, default):
    """sum idf(shared tokens) / sum idf(union): a shared rare token counts more than a shared 'services'."""
    n = len(a_keys)
    out = np.zeros(n, np.float32)
    get = idf.get
    for i in range(n):
        A = a_keys[i]; B = b_keys[i]
        if not A or not B:
            continue
        sa = set(A.split()); sb = set(B.split())
        inter = sa & sb
        if not inter:
            continue
        out[i] = sum(get(t, default) for t in inter) / sum(get(t, default) for t in (sa | sb))
    return out


def numeric_jaccard(a_norm, b_norm):
    n = len(a_norm)
    out = np.zeros(n, np.float32)
    for i in range(n):
        sa = {t for t in a_norm[i].split() if t.isdigit()}
        if not sa:
            continue
        sb = {t for t in b_norm[i].split() if t.isdigit()}
        if sb:
            out[i] = len(sa & sb) / len(sa | sb)
    return out


class FeatureContext:
    """TF-IDF vectorizers + IDF table.  Fitted ONLY on unsupervised text from the training split
    (random pool names, training-split Source-1 names, random pool addresses); restored from the
    artifacts at inference - nothing is refit on test data."""
    def __init__(self, s1_nt=None, pool_nt=None, train_rows_s1=None, fit_rows=None, artifacts=None):
        if artifacts is not None:
            self.tf_char, self.tf_word, self.tf_addr = artifacts['tf_char'], artifacts['tf_word'], artifacts['tf_addr']
            self.idf, self.idf_default = artifacts['idf'], artifacts['idf_default']
            return
        t0 = time.time()
        rs = np.random.RandomState(SEED)
        pool_rows = np.sort(rs.choice(len(pool_nt), min(fit_rows, len(pool_nt)), replace=False))
        s1_rows = np.sort(rs.choice(np.asarray(train_rows_s1), min(fit_rows // 4, len(train_rows_s1)), replace=False))
        names = [x for x in pool_nt.get('name_norm', pool_rows) + s1_nt.get('name_norm', s1_rows) if x]
        addr_rows = np.sort(rs.choice(len(pool_nt), min(fit_rows, len(pool_nt)), replace=False))
        addrs = [x for x in pool_nt.get('addr_norm', addr_rows) if x]
        self.tf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=3, max_features=250_000,
                                       sublinear_tf=True, dtype=np.float32).fit(names)
        self.tf_word = TfidfVectorizer(token_pattern=r'\S+', min_df=2, max_features=250_000, sublinear_tf=True,
                                       dtype=np.float32).fit(names)
        self.tf_addr = TfidfVectorizer(token_pattern=r'\S+', min_df=2, max_features=250_000, sublinear_tf=True,
                                       dtype=np.float32).fit(addrs if addrs else ['empty'])
        self.idf = dict(zip(self.tf_word.get_feature_names_out().tolist(), self.tf_word.idf_.astype(float).tolist()))
        self.idf_default = float(max(self.idf.values()) + 1.0) if self.idf else 1.0
        log(f"FeatureContext fitted on {len(names):,} names / {len(addrs):,} addresses: char vocab "
            f"{len(self.tf_char.vocabulary_):,}, word vocab {len(self.tf_word.vocabulary_):,}, addr vocab "
            f"{len(self.tf_addr.vocabulary_):,} ({time.time() - t0:.1f}s)")


RANK_COLS = ['n_tsr', 'n_tfidf_char', 'pre_score', 'a_jacc']
FEATURE_NAMES = (['name_key_eq', 'name_norm_eq', 'leet_eq', 'n_ratio', 'n_partial', 'n_tsr', 'n_tsort', 'n_wr', 'n_jw',
                  'n_levsim', 'ns_ratio', 'ns_partial', 'ns_eq', 'n_jacc', 'n_cont_s1', 'n_cont_c', 'n_common',
                  'n_idf_overlap', 'n_tfidf_char', 'n_tfidf_word', 'n_ntok1', 'n_ntok2', 'n_len1', 'n_len2', 'n_len_ratio',
                  'first_eq', 'soundex_eq', 'indic1', 'indic2', 'domain2', 'addr_missing', 'addr_key_eq', 'state_known',
                  'state_eq', 'geo_eq', 'city_eq', 'city_jw', 'house_known', 'house_eq', 'house_sim', 'street_eq',
                  'street_jw', 'post_eq', 'a_jacc', 'a_cont_s1', 'a_cont_c', 'a_common', 'a_tsr', 'a_tfidf', 'num_jacc',
                  'src3', 'name_freq_pool', 'name_freq_s1']
                 + ['hit_' + k for k in KEY_TYPES] + ['nhits', 'pre_score', 'ncand']
                 + [f'{c}_{s}' for c in RANK_COLS for s in ('gap', 'rank', 'max')])
assert len(FEATURE_NAMES) == 77 and len(set(FEATURE_NAMES)) == 77


def entity_aligned_chunks(q, max_pairs):
    """[(start, end)] slices of at most ~max_pairs pairs that never split an entity (q must be grouped)."""
    n = len(q)
    if n == 0:
        return []
    bounds = np.r_[0, np.flatnonzero(np.diff(q)) + 1, n]      # entity start offsets + n
    out, start = [], 0
    while start < n:
        target = start + max_pairs
        if target >= n:
            out.append((start, n)); break
        j = np.searchsorted(bounds, target, side='right') - 1
        end = int(bounds[j]) if bounds[j] > start else int(bounds[j + 1])
        out.append((start, end)); start = end
    return out


def build_features(s1_nt, pool_nt, cand, ctx, name_freq_pool, name_freq_s1):
    """cand (q, p, hits, nhits), whole entities only -> float32 DataFrame with the 77 FEATURE_NAMES columns."""
    q = cand['q'].to_numpy(); p = cand['p'].to_numpy(); n = len(q)
    n1 = s1_nt.get('name_norm', q); n2 = pool_nt.get('name_norm', p)
    k1 = s1_nt.get('name_key', q); k2 = pool_nt.get('name_key', p)
    l1 = s1_nt.get('leet_key', q); l2 = pool_nt.get('leet_key', p)
    a1 = s1_nt.get('addr_norm', q); a2 = pool_nt.get('addr_norm', p)
    ak1 = s1_nt.get('addr_key', q); ak2 = pool_nt.get('addr_key', p)
    F = {}
    K1 = np.asarray(k1, dtype=object); K2 = np.asarray(k2, dtype=object)
    F['name_key_eq'] = (K1 == K2)
    F['name_norm_eq'] = (np.asarray(n1, dtype=object) == np.asarray(n2, dtype=object))
    F['leet_eq'] = np.fromiter((float(bool((x and (x == y or x == kb)) or (y and y == ka)))
                                for x, y, ka, kb in zip(l1, l2, k1, k2)), dtype=np.float32, count=n)
    for name, key in [('n_ratio', 'ratio'), ('n_partial', 'partial'), ('n_tsr', 'tsr'), ('n_tsort', 'tsort'),
                      ('n_wr', 'wr'), ('n_jw', 'jw'), ('n_levsim', 'levsim')]:
        F[name] = rf_batch(SC[key], n1, n2)
    ns1 = [x.replace(' ', '') for x in n1]; ns2 = [x.replace(' ', '') for x in n2]      # domain-style concatenations
    F['ns_ratio'] = rf_batch(SC['ratio'], ns1, ns2)
    F['ns_partial'] = rf_batch(SC['partial'], ns1, ns2)
    F['ns_eq'] = (np.asarray(ns1, dtype=object) == np.asarray(ns2, dtype=object))
    F['n_jacc'], F['n_cont_s1'], F['n_cont_c'], F['n_common'] = set_stats(k1, k2)
    F['n_idf_overlap'] = idf_overlap(k1, k2, ctx.idf, ctx.idf_default)
    F['n_tfidf_char'] = tfidf_cosine(ctx.tf_char, n1, n2)
    F['n_tfidf_word'] = tfidf_cosine(ctx.tf_word, n1, n2)
    t1 = [x.split() for x in n1]; t2 = [x.split() for x in n2]
    F['n_ntok1'] = [len(t) for t in t1]; F['n_ntok2'] = [len(t) for t in t2]
    len1 = np.array([len(x) for x in n1], np.float32); len2 = np.array([len(x) for x in n2], np.float32)
    F['n_len1'] = len1; F['n_len2'] = len2
    F['n_len_ratio'] = np.where(np.maximum(len1, len2) > 0, np.minimum(len1, len2) / np.maximum(np.maximum(len1, len2), 1), 0)
    F['first_eq'] = [float(bool(a and b and a[0] == b[0])) for a, b in zip(t1, t2)]
    F['soundex_eq'] = [float(bool(a and b and soundex(a[0]) and soundex(a[0]) == soundex(b[0]))) for a, b in zip(t1, t2)]
    F['indic1'] = s1_nt.get('is_indic', q); F['indic2'] = pool_nt.get('is_indic', p); F['domain2'] = pool_nt.get('is_domain', p)
    has1 = s1_nt.get('has_addr', q).astype(np.float32); has2 = pool_nt.get('has_addr', p).astype(np.float32)
    both_addr = has1 * has2
    F['addr_missing'] = 1.0 - both_addr
    F['addr_key_eq'] = [float(bool(x and x == y)) for x, y in zip(ak1, ak2)]
    st1 = s1_nt.get('state', q); st2 = pool_nt.get('state', p)
    F['state_known'] = [float(bool(x and y)) for x, y in zip(st1, st2)]
    F['state_eq'] = [float(bool(x and x == y)) for x, y in zip(st1, st2)]
    g1 = s1_nt.get('geo', q); g2 = pool_nt.get('geo', p)
    F['geo_eq'] = [float(bool(x and y and (set(x.split()) & set(y.split())))) for x, y in zip(g1, g2)]
    c1 = s1_nt.get('city', q); c2 = pool_nt.get('city', p)
    cboth = np.array([float(bool(x and y)) for x, y in zip(c1, c2)], np.float32)
    F['city_eq'] = [float(bool(x and x == y)) for x, y in zip(c1, c2)]
    F['city_jw'] = rf_batch(SC['jw'], c1, c2) * cboth
    h1 = s1_nt.get('house', q); h2 = pool_nt.get('house', p)
    hboth = np.array([float(bool(x and y)) for x, y in zip(h1, h2)], np.float32)
    F['house_known'] = hboth
    F['house_eq'] = [float(bool(x and x == y)) for x, y in zip(h1, h2)]
    F['house_sim'] = rf_batch(SC['levsim'], h1, h2) * hboth
    s1s = s1_nt.get('street', q); s2s = pool_nt.get('street', p)
    sboth = np.array([float(bool(x and y)) for x, y in zip(s1s, s2s)], np.float32)
    F['street_eq'] = [float(bool(x and x == y)) for x, y in zip(s1s, s2s)]
    F['street_jw'] = rf_batch(SC['jw'], s1s, s2s) * sboth
    pc1 = s1_nt.get('postcode', q); pc2 = pool_nt.get('postcode', p)
    F['post_eq'] = [float(bool(x and x == y)) for x, y in zip(pc1, pc2)]
    F['a_jacc'], F['a_cont_s1'], F['a_cont_c'], F['a_common'] = set_stats(ak1, ak2)
    F['a_tsr'] = rf_batch(SC['tsr'], a1, a2) * both_addr
    F['a_tfidf'] = tfidf_cosine(ctx.tf_addr, a1, a2)
    F['num_jacc'] = numeric_jaccard(a1, a2)
    F['src3'] = [float(x.startswith('S3')) for x in pool_nt.get('entity_id', p)]
    keys = [f"{c}|{k}" for c, k in zip(s1_nt.get('country', q), k1)]
    F['name_freq_pool'] = np.log1p(name_freq_pool.lookup(keys).astype(np.float32))
    F['name_freq_s1'] = np.log1p(name_freq_s1.lookup(keys).astype(np.float32))
    hits = cand['hits'].to_numpy().astype(np.int32)
    for k in KEY_TYPES:
        F['hit_' + k] = (hits & KEY_BIT[k]) > 0
    F['nhits'] = cand['nhits'].to_numpy()
    F['pre_score'] = F['n_tsr'] / 100.0 + F['n_tfidf_char'] + F['a_tsr'] / 100.0 + F['a_jacc']
    df = pd.DataFrame({k: np.asarray(v, dtype=np.float32) for k, v in F.items()})
    # entity-relative features: rank / gap / max within the entity's own candidate list (whole entities per chunk)
    grp = df.groupby(q, sort=False)
    df['ncand'] = np.log1p(grp['n_tsr'].transform('size').to_numpy().astype(np.float32))
    for col in RANK_COLS:
        mx = grp[col].transform('max').to_numpy().astype(np.float32)
        df[col + '_gap'] = mx - df[col].to_numpy()
        df[col + '_rank'] = grp[col].rank(method='min', ascending=False).to_numpy().astype(np.float32)
        df[col + '_max'] = mx
    return df[FEATURE_NAMES].astype(np.float32)

## 8. Training and validation sets — split by Source-1 entity

The labelled Source-1 entities are shuffled with `RandomState(42)`; the first `N_VAL_S1` become the validation set and the next `N_TRAIN_S1` the training set. Every candidate produced by blocking is labelled from the ground truth: positives are the blocked true pairs, negatives every other blocked candidate — the hard negatives the model faces at inference. No random negatives and no unblocked positives are added, so the training distribution equals the inference distribution. Blocking recall is reported at pair level and at entity level (all matches of an entity retrieved) against the full ground truth.

**Leakage rules honoured by the code:** the split is by entity, never by pair; validation labels are used only for early stopping, threshold selection and reporting; the transliteration map, the TF-IDF vocabularies and the IDF weights are fitted on training-split data only; rank features use only the entity's own candidates; nothing from the test files is read before the inference cell, and the test files are never used to fit anything.

In [ ]:
# =====================================================================================
# Cell 10 - training / validation sets: split by Source-1 entity, block, label, featurize
# =====================================================================================
T_TRAINPREP = time.time()
labeled = np.array([x for x in S1['entity_id'].tolist() if x in GT], dtype=object)
rs_split = np.random.RandomState(SEED)
rs_split.shuffle(labeled)
n_val = min(CONFIG['N_VAL_S1'], len(labeled) // 4)
VAL_IDS = labeled[:n_val].tolist()
TRAIN_IDS = labeled[n_val:n_val + CONFIG['N_TRAIN_S1']].tolist()
N_LABELED = len(labeled)
del labeled
log(f"labeled Source-1 entities: {N_LABELED:,} | validation {len(VAL_IDS):,} | training {len(TRAIN_IDS):,} "
    f"(split BY ENTITY; validation labels are used only for early stopping, threshold selection and reporting)")

TRANSLIT = learn_translit_map(S1, POOL, GT, TRAIN_IDS, s1_pos=S1_POS, pool_pos=POOL_POS)
S1N = normalize_table(S1, TRANSLIT, 'S1')
PN = normalize_table(POOL, TRANSLIT, 'POOL')
BI = BlockIndex(PN, CONFIG['BLOCK_CAPS'], 'train pool')
NAME_FREQ_S1 = KeyCounter.from_keys([f"{c}|{k}" for c, k in zip(S1N.get('country'), S1N.get('name_key'))])
TRAIN_ROWS = S1_POS.get_indexer(TRAIN_IDS)
CTX = FeatureContext(S1N, PN, TRAIN_ROWS, CONFIG['TFIDF_FIT_ROWS'])
gc.collect()


def gt_pairs(ids):
    """Vectorized (S1 row, pool row) true pairs for the given Source-1 ids (GT ids absent from the pool are
    warned about and dropped from the pair list, but still counted in the recall denominators)."""
    s1_list, p_list = [], []
    for s1_id in ids:
        for pid in GT.get(s1_id, ()):
            s1_list.append(s1_id); p_list.append(pid)
    if not s1_list:
        return np.zeros(0, np.int64), np.zeros(0, np.int64)
    si = S1_POS.get_indexer(s1_list); pi = POOL_POS.get_indexer(p_list)
    ok = (si >= 0) & (pi >= 0)
    if (~ok).sum():
        log(f"WARNING: {int((~ok).sum()):,} ground-truth ids are absent from the S2/S3 files")
    return si[ok].astype(np.int64), pi[ok].astype(np.int64)


def pair_code(q, p):
    return (np.asarray(q, np.int64) << 32) | np.asarray(p, np.int64)


def block_entities(bi, nt_q, rows, chunk, max_cand, label):
    parts = []
    starts = list(range(0, len(rows), chunk))
    prog = progress_for(len(starts), f"blocking {label}")
    for s in starts:
        parts.append(bi.query(nt_q, rows[s:s + chunk], max_cand))
        prog.update(1)
    prog.close()
    return pd.concat(parts, ignore_index=True) if parts else bi.query(nt_q, rows[:0], max_cand)


def featurize(s1_nt, pool_nt, cand, ctx, nf_pool, nf_s1, label):
    """Entity-aligned PAIR_CHUNK chunks written into ONE preallocated float32 matrix (no concat doubling)."""
    slices = entity_aligned_chunks(cand['q'].to_numpy(), CONFIG['PAIR_CHUNK'])
    X = np.empty((len(cand), len(FEATURE_NAMES)), dtype=np.float32)
    prog = progress_for(len(slices), f"features {label}")
    for s, e in slices:
        X[s:e] = build_features(s1_nt, pool_nt, cand.iloc[s:e], ctx, nf_pool, nf_s1).to_numpy()
        prog.update(1)
    prog.close()
    return pd.DataFrame(X, columns=FEATURE_NAMES, copy=False)


def make_dataset(ids, label):
    """Block the Source-1 entities `ids` against the full pool and label every candidate pair from the GT.
    Positives = blocked true pairs; negatives = every other blocked candidate (the hard negatives the model
    faces at inference).  No random negatives and no unblocked positives are added, so the training
    distribution equals the inference distribution."""
    t0 = time.time()
    rows = S1_POS.get_indexer(ids)
    assert (rows >= 0).all()
    cand = block_entities(BI, S1N, rows, CONFIG['CHUNK_S1'], CONFIG['MAX_CAND_PER_S1'], label)
    gq, gp = gt_pairs(ids)
    gt_codes = pair_code(gq, gp)
    codes = pair_code(cand['q'].to_numpy(), cand['p'].to_numpy())
    y = np.isin(codes, gt_codes).astype(np.int8)
    n_true = len(gt_codes)
    pair_recall = float(np.isin(gt_codes, codes).mean()) if n_true else float('nan')
    # entity-level recall: all matches of the entity retrieved (over entities with >= 1 match)
    found_per_entity = pd.Series(np.isin(gt_codes, codes)).groupby(gq).all() if n_true else pd.Series(dtype=bool)
    ent_recall = float(found_per_entity.mean()) if n_true else float('nan')
    n_singletons = int(sum(1 for i in ids if not GT.get(i)))
    log(f"{label}: {len(ids):,} entities -> {len(cand):,} candidate pairs ({len(cand) / max(1, len(ids)):.1f} per entity) | "
        f"positives {int(y.sum()):,} | reduction ratio {len(ids) * len(POOL) / max(1, len(cand)):,.0f}x | "
        f"blocking recall: pairs {100 * pair_recall:.2f}%, entities (all matches) {100 * ent_recall:.2f}% | "
        f"singletons {n_singletons:,} ({100 * n_singletons / max(1, len(ids)):.1f}%) | {fmt_secs(time.time() - t0)}")
    X = featurize(S1N, PN, cand, CTX, BI.name_freq, NAME_FREQ_S1, label)
    log(f"{label}: feature matrix {X.shape} ({X.memory_usage().sum() / 1e6:.0f} MB) | {fmt_secs(time.time() - t0)}")
    return cand, X, y, (gq, gp)


cand_tr, X_tr, y_tr, gt_tr = make_dataset(TRAIN_IDS, 'train')
cand_va, X_va, y_va, gt_va = make_dataset(VAL_IDS, 'validation')
VAL_ROWS = S1_POS.get_indexer(VAL_IDS)
gc.collect()
log(f"training data ready in {fmt_secs(time.time() - T_TRAINPREP)}: X_tr {X_tr.shape}, X_va {X_va.shape}, "
    f"positive rate train {y_tr.mean():.4f} / val {y_va.mean():.4f}")

## 9. Model

A LightGBM binary classifier (`learning_rate 0.05`, `num_leaves 63`, `min_data_in_leaf 40`, feature and bagging fractions 0.8, `lambda_l2 1.0`, seed 42) is trained for up to `LGB_ROUNDS` rounds with early stopping (100 rounds) on the validation pairs; a progress line is printed every 1% of the rounds with the current validation log-loss. Predictions always use `best_iteration`. Without LightGBM the notebook falls back to scikit-learn's `HistGradientBoostingClassifier`. The model is MIT-licensed and far below the 8B-parameter limit.

In [ ]:
# =====================================================================================
# Cell 11 - pair classifier: LightGBM with early stopping (HistGradientBoosting fallback)
# =====================================================================================
from sklearn.metrics import roc_auc_score, average_precision_score
T_MODEL = time.time()
LGB_PARAMS = dict(objective='binary', learning_rate=0.05, num_leaves=63, min_data_in_leaf=40, feature_fraction=0.8,
                  bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, num_threads=N_JOBS, seed=SEED, verbose=-1)


def lgb_progress(total_rounds):
    """Progress line every 1% of LGB_ROUNDS: percent, last-1% time, elapsed, ETA and the current valid logloss."""
    step = max(1, total_rounds // 100)
    state = dict(t0=time.time(), t_last=time.time())

    def _cb(env):
        it = env.iteration + 1
        if it % step == 0 or it == total_rounds:
            now = time.time(); pct = 100.0 * it / total_rounds
            elapsed = now - state['t0']; eta = elapsed / it * (total_rounds - it)
            ll = float('nan')
            for item in (env.evaluation_result_list or []):
                if item[1] in ('binary_logloss', 'logloss'):
                    ll = item[2]; break
            print(f"[lightgbm] {pct:6.2f}% | last 1%: {now - state['t_last']:6.1f}s | elapsed: {fmt_secs(elapsed)} | "
                  f"ETA: {fmt_secs(eta)} | valid logloss: {ll:.5f}", flush=True)
            state['t_last'] = now
    _cb.order = 30
    return _cb


def _X(X):
    """Feature matrix in FEATURE_NAMES order (no copy when the columns already match)."""
    return X if list(X.columns) == FEATURE_NAMES else X[FEATURE_NAMES]


if HAVE_LGB:
    dtr = lgb.Dataset(_X(X_tr), label=y_tr, feature_name=FEATURE_NAMES, free_raw_data=True)
    dva = lgb.Dataset(_X(X_va), label=y_va, reference=dtr)
    MODEL = lgb.train(LGB_PARAMS, dtr, num_boost_round=CONFIG['LGB_ROUNDS'], valid_sets=[dva], valid_names=['valid'],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb_progress(CONFIG['LGB_ROUNDS'])])
    BEST_ITER = int(MODEL.best_iteration) if MODEL.best_iteration else int(MODEL.current_iteration())

    def predict(X):
        return MODEL.predict(_X(X), num_iteration=BEST_ITER)

    IMP = pd.DataFrame({'feature': FEATURE_NAMES, 'gain': MODEL.feature_importance('gain'),
                        'split': MODEL.feature_importance('split')})
    IMP['gain_norm'] = IMP['gain'] / max(1e-12, IMP['gain'].sum())
    del dtr, dva
else:
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.inspection import permutation_importance
    MODEL = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, max_leaf_nodes=63, early_stopping=True,
                                           random_state=SEED).fit(_X(X_tr), y_tr)
    BEST_ITER = int(MODEL.n_iter_)

    def predict(X):
        return MODEL.predict_proba(_X(X))[:, 1]

    _sub = np.random.RandomState(SEED).choice(len(X_va), min(20_000, len(X_va)), replace=False)
    _pi = permutation_importance(MODEL, _X(X_va).iloc[_sub], y_va[_sub], n_repeats=1, random_state=SEED, n_jobs=N_JOBS)
    IMP = pd.DataFrame({'feature': FEATURE_NAMES, 'gain': np.maximum(_pi.importances_mean, 0), 'split': 0})
    IMP['gain_norm'] = IMP['gain'] / max(1e-12, IMP['gain'].sum())
IMP = IMP.sort_values('gain_norm', ascending=False).reset_index(drop=True)
IMP.to_csv(os.path.join(ARTIFACT_DIR, 'feature_importance.csv'), index=False)
P_VA = predict(X_va)
VAL_AUC = float(roc_auc_score(y_va, P_VA)) if 0 < y_va.sum() < len(y_va) else float('nan')
VAL_AP = float(average_precision_score(y_va, P_VA)) if 0 < y_va.sum() < len(y_va) else float('nan')
log(f"model: {'LightGBM' if HAVE_LGB else 'HistGradientBoosting'} | best iteration {BEST_ITER} | "
    f"validation pair-level ROC-AUC {VAL_AUC:.5f} | average precision {VAL_AP:.5f} | {fmt_secs(time.time() - T_MODEL)}")
print("\nTop-25 features by normalised gain:")
print(IMP.head(25).to_string(index=False))

## 10. Decision rule

The leaderboard metric is entity-level, so the pair probabilities are turned into sets with a rule chosen on the validation entities: `accept = p ≥ t`, optionally extended by `p ≥ t_lo` **and** `p ≥ (entity maximum − margin)`. All 76 thresholds between 0.20 and 0.95 are tried for the plain rule and for margins 0.05 / 0.10; the macro F0.5 is computed over **all** validation entities, including those without candidates and matches that were never blocked (their true matches count in the recall denominator). Ties are broken towards the larger threshold.

In [ ]:
# =====================================================================================
# Cell 12 - decision rule: threshold (+ optional entity-relative margin) maximising macro F0.5
# =====================================================================================
T_THR = time.time()


def entity_max(prob, ent, n_ent):
    m = np.full(n_ent, -np.inf)
    np.maximum.at(m, ent, prob)
    return m[ent]


def decide(q, prob, t, margin=None, t_lo=None):
    """accept = prob >= t, optionally also prob >= t_lo AND within `margin` of the entity's best candidate
    (an entity's second-best record with a near-top score is usually a further true duplicate)."""
    prob = np.asarray(prob)
    acc = prob >= t
    if margin is not None and len(prob):
        ent, uniq = pd.factorize(np.asarray(q))
        acc = acc | ((prob >= t_lo) & (prob >= entity_max(prob, ent, len(uniq)) - margin))
    return acc


def entity_scores(ent, y, accept, nt, n_ent):
    """Per-entity F0.5 over ALL entities (those without candidates included): tp = accepted true pairs,
    npred = accepted pairs, nt = true matches from the GT (never-blocked ones included)."""
    y = np.asarray(y).astype(bool); accept = np.asarray(accept).astype(bool)
    tp = np.bincount(ent[accept & y], minlength=n_ent).astype(float)
    npred = np.bincount(ent[accept], minlength=n_ent).astype(float)
    prec = np.where(npred > 0, tp / np.maximum(npred, 1), 0.0)
    rec = np.where(nt > 0, tp / np.maximum(nt, 1), 0.0)
    f = np.where(nt == 0, (npred == 0).astype(float),
                 np.where(tp == 0, 0.0, 1.25 * prec * rec / np.maximum(0.25 * prec + rec, 1e-12)))
    return f, prec, rec, tp, npred


N_ENT_VA = len(VAL_IDS)
ENT_VA = pd.Index(VAL_ROWS).get_indexer(cand_va['q'].to_numpy())          # local entity index per pair
assert (ENT_VA >= 0).all()
NT_VA = np.array([len(GT.get(i, ())) for i in VAL_IDS], dtype=float)      # true matches per validation entity
Q_VA = cand_va['q'].to_numpy()

THRESHOLDS = np.round(np.arange(0.20, 0.96, 0.01), 2)
RULES = [('plain', None, None), ('margin 0.05', 0.05, lambda t: max(0.2, t - 0.15)), ('margin 0.10', 0.10, lambda t: max(0.2, t - 0.20))]
rows = []
prog = progress_for(len(THRESHOLDS) * len(RULES), 'threshold grid')
for rule_name, margin, tlo_fn in RULES:
    for t in THRESHOLDS:
        t_lo = float(tlo_fn(t)) if tlo_fn else None
        acc = decide(Q_VA, P_VA, float(t), margin, t_lo)
        f, prec, rec, tp, npred = entity_scores(ENT_VA, y_va, acc, NT_VA, N_ENT_VA)
        rows.append(dict(rule=rule_name, t=float(t), margin=margin, t_lo=t_lo, macro_f05=float(f.mean()),
                         pair_precision=float(tp.sum() / max(1.0, npred.sum())), pair_recall=float(tp.sum() / max(1.0, NT_VA.sum())),
                         avg_predicted=float(npred.mean())))
        prog.update(1)
prog.close()
GRID = pd.DataFrame(rows)
GRID.to_csv(os.path.join(ARTIFACT_DIR, 'threshold_grid.csv'), index=False)
best = GRID.sort_values(['macro_f05', 't'], ascending=[False, False]).iloc[0]
RULE = dict(t=float(best['t']), margin=(None if pd.isna(best['margin']) else float(best['margin'])),
            t_lo=(None if pd.isna(best['t_lo']) else float(best['t_lo'])))
plain = GRID[GRID['rule'] == 'plain'].reset_index(drop=True)
print("plain rule, every 5th threshold:")
print(plain.iloc[::5][['t', 'macro_f05', 'pair_precision', 'pair_recall', 'avg_predicted']].to_string(index=False))
print(f"\nbest per rule:\n{GRID.loc[GRID.groupby('rule')['macro_f05'].idxmax()][['rule', 't', 't_lo', 'macro_f05', 'pair_precision', 'pair_recall']].to_string(index=False)}")
log(f"chosen rule: {RULE} -> validation macro F0.5 {best['macro_f05']:.5f} ({fmt_secs(time.time() - T_THR)})")
fig, ax = plt.subplots(figsize=(7, 3.4))
for rule_name, sub in GRID.groupby('rule'):
    ax.plot(sub['t'], sub['macro_f05'], label=f'macro F0.5 ({rule_name})')
ax.plot(plain['t'], plain['pair_precision'], '--', color='grey', label='pair precision (plain)')
ax.plot(plain['t'], plain['pair_recall'], ':', color='grey', label='pair recall (plain)')
ax.axvline(RULE['t'], color='red', lw=0.8); ax.set_xlabel('threshold t'); ax.set_ylim(0, 1); ax.legend(fontsize=7); ax.set_title('validation: macro F0.5 / precision / recall vs t')
plt.tight_layout(); plt.savefig(os.path.join(ARTIFACT_DIR, 'threshold_curve.png'), dpi=90); plt.show(); plt.close(fig)

## 11. Evaluation

With the chosen rule: macro F0.5 (the leaderboard metric), the pair-level micro precision / recall / F0.5, the oracle macro F0.5 of a perfect classifier on the blocked candidates (the ceiling set by blocking), singleton statistics, exact-set and any-match rates for non-singletons, breakdowns by number of true matches, by country (string labels) and by candidate source, and the predicted-vs-true cross-tab. Everything is saved to `artifacts/validation_metrics.json`.

In [ ]:
# =====================================================================================
# Cell 13 - validation report with the chosen rule (the leaderboard metric is macro F0.5)
# =====================================================================================
ACC_VA = decide(Q_VA, P_VA, RULE['t'], RULE['margin'], RULE['t_lo'])
F_VA, PREC_VA, REC_VA, TP_VA, NPRED_VA = entity_scores(ENT_VA, y_va, ACC_VA, NT_VA, N_ENT_VA)
MACRO_F05 = float(F_VA.mean())
tp_sum, pred_sum, nt_sum = float(TP_VA.sum()), float(NPRED_VA.sum()), float(NT_VA.sum())
MICRO_P = tp_sum / max(1.0, pred_sum); MICRO_R = tp_sum / max(1.0, nt_sum)
MICRO_F05 = 1.25 * MICRO_P * MICRO_R / max(1e-12, 0.25 * MICRO_P + MICRO_R)
f_or, *_ = entity_scores(ENT_VA, y_va, y_va.astype(bool), NT_VA, N_ENT_VA)       # perfect classifier on blocked pairs
ORACLE_F05 = float(f_or.mean())
is_single = NT_VA == 0
METRICS = dict(
    n_entities=int(N_ENT_VA), rule=RULE, macro_f05=MACRO_F05, oracle_macro_f05=ORACLE_F05,
    pair_precision=MICRO_P, pair_recall=MICRO_R, pair_f05=MICRO_F05, roc_auc=VAL_AUC, average_precision=VAL_AP,
    singleton_share=float(is_single.mean()),
    singletons_left_empty=float((NPRED_VA[is_single] == 0).mean()) if is_single.any() else float('nan'),
    false_merges_on_singletons=int(NPRED_VA[is_single].sum()),
    nonsingleton_exact_set=float(((TP_VA == NT_VA) & (NPRED_VA == TP_VA))[~is_single].mean()) if (~is_single).any() else float('nan'),
    nonsingleton_any_found=float((TP_VA[~is_single] > 0).mean()) if (~is_single).any() else float('nan'),
)
print(f"macro F0.5 (leaderboard metric): {MACRO_F05:.5f}   | oracle macro F0.5 on blocked candidates: {ORACLE_F05:.5f}")
print(f"pair-level micro precision {MICRO_P:.4f} | recall {MICRO_R:.4f} | F0.5 {MICRO_F05:.4f}")
print(f"singleton share {METRICS['singleton_share']:.4f} | true singletons left empty {METRICS['singletons_left_empty']:.4f} | "
      f"false merges on singletons {METRICS['false_merges_on_singletons']}")
print(f"non-singletons: exact set recovered {METRICS['nonsingleton_exact_set']:.4f} | at least one match found {METRICS['nonsingleton_any_found']:.4f}")
bucket = np.minimum(NT_VA, 4).astype(int)
BY_NT = pd.DataFrame({'true_matches': ['0', '1', '2', '3', '4+'][:0]})
rows = []
for b, lab in enumerate(['0', '1', '2', '3', '4+']):
    m = bucket == b
    if m.any():
        rows.append(dict(true_matches=lab, entities=int(m.sum()), macro_f05=float(F_VA[m].mean()), precision=float(PREC_VA[m].mean()),
                         recall=float(REC_VA[m].mean()), avg_predicted=float(NPRED_VA[m].mean())))
BY_NT = pd.DataFrame(rows)
print("\nby number of true matches:\n" + BY_NT.to_string(index=False))
VAL_COUNTRY = np.asarray(S1N.get('country', VAL_ROWS), dtype=object)
BY_COUNTRY = pd.DataFrame({'country': VAL_COUNTRY, 'f05': F_VA}).groupby('country')['f05'].agg(['size', 'mean']).rename(columns={'size': 'entities', 'mean': 'macro_f05'}).reset_index()
print("\nby country (string labels, open set):\n" + BY_COUNTRY.to_string(index=False))
src3 = X_va['src3'].to_numpy() > 0.5
rows = []
for lab, m in [('S2', ~src3), ('S3', src3)]:
    if m.any():
        tp_ = float((ACC_VA & (y_va == 1) & m).sum()); pred_ = float((ACC_VA & m).sum()); pos_ = float(((y_va == 1) & m).sum())
        rows.append(dict(source=lab, blocked_pairs=int(m.sum()), positives=int(pos_), precision=tp_ / max(1.0, pred_), recall_among_blocked=tp_ / max(1.0, pos_)))
BY_SRC = pd.DataFrame(rows)
print("\nper candidate source (among blocked pairs):\n" + BY_SRC.to_string(index=False))
XTAB = pd.crosstab(pd.Series(np.minimum(NPRED_VA, 4).astype(int), name='predicted'), pd.Series(np.minimum(NT_VA, 4).astype(int), name='true'))
print("\npredicted vs true match counts (capped at 4):\n" + XTAB.to_string())
METRICS.update(by_true_matches=BY_NT.to_dict(orient='records'), by_country=BY_COUNTRY.to_dict(orient='records'),
               by_source=BY_SRC.to_dict(orient='records'), crosstab=XTAB.to_dict(),
               blocking=dict(train_pairs=int(len(cand_tr)), val_pairs=int(len(cand_va)), max_cand_per_s1=CONFIG['MAX_CAND_PER_S1']))
with open(os.path.join(ARTIFACT_DIR, 'validation_metrics.json'), 'w') as fh:
    json.dump(METRICS, fh, indent=2, default=lambda o: (o.item() if hasattr(o, 'item') else str(o)))
log(f"validation metrics saved to {os.path.join(ARTIFACT_DIR, 'validation_metrics.json')}")

## 12. Error analysis

(a) false merges — accepted pairs that are not true matches, the most confident ones printed raw; (b) missed matches, split into pairs that blocking never produced and pairs the model rejected; (c) ambiguous pairs whose probability lies within 0.10 of the threshold.

In [ ]:
# =====================================================================================
# Cell 14 - error analysis on the validation split
# =====================================================================================
def show_pair(q_row, p_row, extra=''):
    print(f"   S1   [{S1N.get('entity_id', [q_row])[0]}] {S1N.get('raw_name', [q_row])[0]} | "
          f"{S1N.get('raw_addr', [q_row])[0]} | {S1N.get('country', [q_row])[0]}")
    print(f"   cand [{PN.get('entity_id', [p_row])[0]}] {PN.get('raw_name', [p_row])[0]} | "
          f"{PN.get('raw_addr', [p_row])[0]}   {extra}")


P_VA_ARR = np.asarray(P_VA); QV = cand_va['q'].to_numpy(); PV = cand_va['p'].to_numpy()
# (a) FALSE MERGES: accepted pairs that are not true matches
fm = np.flatnonzero(ACC_VA & (y_va == 0))
print(f"(a) FALSE MERGES: {len(fm):,} of {int(ACC_VA.sum()):,} accepted pairs ({100.0 * len(fm) / max(1, ACC_VA.sum()):.2f}%)")
for i in fm[np.argsort(-P_VA_ARR[fm])][:6]:
    show_pair(QV[i], PV[i], f"p={P_VA_ARR[i]:.3f} n_tsr={X_va['n_tsr'].iat[i]:.0f} a_jacc={X_va['a_jacc'].iat[i]:.2f} name_freq_pool={X_va['name_freq_pool'].iat[i]:.2f}")
# (b) MISSED MATCHES: never blocked vs blocked-but-rejected
gq, gp = gt_va
codes_va = pair_code(QV, PV)
gt_codes = pair_code(gq, gp)
never = ~np.isin(gt_codes, codes_va)
rejected = np.flatnonzero((y_va == 1) & ~ACC_VA)
print(f"\n(b) MISSED MATCHES: {int(NT_VA.sum() - TP_VA.sum()):,} total = {int(never.sum()):,} never blocked + {len(rejected):,} blocked but rejected")
print("   never blocked (5 examples):")
for j in np.flatnonzero(never)[:5]:
    show_pair(gq[j], gp[j])
print("   blocked but rejected (5 examples):")
for i in rejected[np.argsort(-P_VA_ARR[rejected])][:5]:
    show_pair(QV[i], PV[i], f"p={P_VA_ARR[i]:.3f} n_tsr={X_va['n_tsr'].iat[i]:.0f} a_jacc={X_va['a_jacc'].iat[i]:.2f}")
# (c) AMBIGUOUS: probabilities within 0.10 of the threshold
amb = np.flatnonzero(np.abs(P_VA_ARR - RULE['t']) <= 0.10)
print(f"\n(c) AMBIGUOUS |p - t| <= 0.10: {len(amb):,} pairs ({100.0 * len(amb) / max(1, len(P_VA_ARR)):.2f}% of scored pairs), "
      f"{100.0 * y_va[amb].mean() if len(amb) else 0:.1f}% true matches, {len(np.unique(QV[amb])):,} entities covered")
for i in np.random.RandomState(SEED).choice(amb, min(5, len(amb)), replace=False) if len(amb) else []:
    show_pair(QV[i], PV[i], f"p={P_VA_ARR[i]:.3f} label={int(y_va[i])}")

## 13. Inference on the test set

The test files are opened only here. Both test frames are normalised with the **saved** transliteration map, the blocking index is built on the test pool, Source-1 name-key frequencies are counted on the test Source-1 file (an unsupervised count over the inference universe), and the Source-1 rows are processed in chunks: query → entity-aligned feature chunks → predict → decide. For every Source-1 entity the full capped candidate list and the accepted subset are kept. `write_submission` writes one row per test Source-1 entity in file order, ids sorted and de-duplicated, matches restricted to candidates, empty second column when there is nothing, tab-separated without quoting. When the test files are missing the validation split is written instead and labelled as such.

In [ ]:
# =====================================================================================
# Cell 15 - inference on the test set (test files are opened ONLY here; nothing is fitted on them)
# =====================================================================================
def run_inference(s1_df, pool_df, translit, ctx, predict_fn, rule, caps, max_cand, chunk, label,
                  s1_nt=None, pool_nt=None):
    """Normalize both frames with the SAVED transliteration map, build the blocking index on the pool,
    count Source-1 name keys (unsupervised), then per CHUNK_S1 Source-1 rows: query -> entity-aligned
    feature chunks -> predict -> decide.  Returns (s1_ids, cand_lists, match_lists, pool_nt): per Source-1
    row the full capped candidate id list (sorted, de-duplicated) and the accepted subset."""
    t0 = time.time()
    if s1_nt is None:
        s1_nt = normalize_table(s1_df, translit, f'{label} S1')
    if pool_nt is None:
        pool_nt = normalize_table(pool_df, translit, f'{label} pool')
    bi = BlockIndex(pool_nt, caps, f'{label} pool')
    nf_s1 = KeyCounter.from_keys([f"{c}|{k}" for c, k in zip(s1_nt.get('country'), s1_nt.get('name_key'))])
    n = len(s1_nt)
    cand_lists = [''] * n
    match_lists = [''] * n
    n_pairs = n_acc = 0
    starts = list(range(0, n, chunk))
    prog = progress_for(len(starts), f"inference {label}")
    for s in starts:
        rows = np.arange(s, min(n, s + chunk), dtype=np.int32)
        cand = bi.query(s1_nt, rows, max_cand)
        if len(cand):
            probs = []
            for a, b in entity_aligned_chunks(cand['q'].to_numpy(), CONFIG['PAIR_CHUNK']):
                X = build_features(s1_nt, pool_nt, cand.iloc[a:b], ctx, bi.name_freq, nf_s1)
                probs.append(np.asarray(predict_fn(X), dtype=np.float64))
            prob = np.concatenate(probs)
            acc = decide(cand['q'].to_numpy(), prob, rule['t'], rule.get('margin'), rule.get('t_lo'))
            q = cand['q'].to_numpy(); pids = pool_nt.get('entity_id', cand['p'].to_numpy())
            bounds = np.r_[0, np.flatnonzero(np.diff(q)) + 1, len(q)]
            for i in range(len(bounds) - 1):
                a, b = int(bounds[i]), int(bounds[i + 1])
                ids = sorted(set(pids[a:b]))
                acc_ids = sorted({pid for pid, ok in zip(pids[a:b], acc[a:b]) if ok})
                cand_lists[int(q[a])] = ','.join(ids)
                match_lists[int(q[a])] = ','.join(acc_ids)
            n_pairs += len(cand); n_acc += int(acc.sum())
            del X, probs, prob, cand
        prog.update(1)
    prog.close()
    del bi
    gc.collect()
    log(f"inference {label}: {n:,} Source-1 entities | {n_pairs:,} scored candidate pairs | {n_acc:,} accepted | "
        f"{fmt_secs(time.time() - t0)}")
    return s1_nt.get('entity_id'), cand_lists, match_lists, pool_nt


def _id_string(x):
    if isinstance(x, str):
        return x
    return ','.join(sorted(set(x)))


def write_submission(s1_ids, cand_lists, match_lists, out_dir):
    """One row per Source-1 id in test_source1 order; ids sorted and de-duplicated; matches = accepted AND
    candidate; empty second column when there is nothing.  No quoting (ids contain no special characters)."""
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()
    seen = set(); ids, cands, matches = [], [], []
    for sid, c, m in zip(s1_ids, cand_lists, match_lists):
        if sid in seen:
            continue
        seen.add(sid)
        c_str = _id_string(c); m_str = _id_string(m)
        if m_str and c_str:
            cset = set(c_str.split(','))
            m_str = ','.join(x for x in m_str.split(',') if x in cset)
        elif not c_str:
            m_str = ''
        ids.append(sid); cands.append(c_str); matches.append(m_str)
    mres = pd.DataFrame({'source1_entity_id': ids, 'matched_entity_ids': matches})
    cres = pd.DataFrame({'source1_entity_id': ids, 'candidate_entity_ids': cands})
    mpath = os.path.join(out_dir, 'matching_results.tsv'); cpath = os.path.join(out_dir, 'candidate_pairs.tsv')
    for df, path in [(mres, mpath), (cres, cpath)]:
        df.to_csv(path, sep='\t', index=False, quoting=csv.QUOTE_NONE, escapechar='\\', lineterminator='\n', encoding='utf-8')
    n_with = int((mres['matched_entity_ids'] != '').sum())
    n_matched = int(sum(len(m.split(',')) for m in matches if m)); n_cand = int(sum(len(c.split(',')) for c in cands if c))
    print(f"Source-1 entities: {len(ids):,} | with >= 1 match: {n_with:,} ({100.0 * n_with / max(1, len(ids)):.2f}%) | "
          f"matched ids: {n_matched:,} | candidate ids: {n_cand:,} | written in {time.time() - t0:.1f}s")
    print(f"--- {mpath} (first 400 chars) ---")
    with open(mpath, encoding='utf-8') as fh:
        print(fh.read(400))
    return mres, cres, mpath, cpath


T_INFER = time.time()
SUBMISSION_IS_VALIDATION = not HAVE_TEST
if HAVE_TEST:
    # free the training pool structures first: the test universe is as large as the training one
    del BI, PN, POOL, POOL_POS, cand_tr, X_tr, y_tr, X_va, cand_va
    gc.collect()
    T1 = read_tsv(PATHS['test_source1.tsv'], 'test_source1')
    T2 = read_tsv(PATHS['test_source2.tsv'], 'test_source2')
    T3 = read_tsv(PATHS['test_source3.tsv'], 'test_source3')
    TPOOL = pd.concat([T2, T3], ignore_index=True)
    del T2, T3
    gc.collect()
    log(f"test universe: S1 {len(T1):,} rows | pool {len(TPOOL):,} rows | countries (S1): "
        f"{T1['country'].value_counts().to_dict()}")
    if T1['entity_id'].duplicated().any():
        log(f"WARNING: {int(T1['entity_id'].duplicated().sum())} duplicated ids in test_source1 -> first occurrence kept")
    SUB_IDS, SUB_CANDS, SUB_MATCHES, TPN = run_inference(
        T1, TPOOL, TRANSLIT, CTX, predict, RULE, CONFIG['BLOCK_CAPS'], CONFIG['MAX_CAND_PER_S1'], CONFIG['CHUNK_S1'], 'test')
    OUT_LABEL = 'TEST submission'
else:
    log("test files not found -> writing the VALIDATION split in the challenge format (labelled as such)")
    VAL_S1 = S1.iloc[VAL_ROWS].reset_index(drop=True)
    _val_nt = NormTable(VAL_S1, {c: S1N.cols[c].iloc[VAL_ROWS].tolist() for c in STR_COLS},
                        {c: S1N.flags[c][VAL_ROWS] for c in FLAG_COLS}, 'validation S1')
    SUB_IDS, SUB_CANDS, SUB_MATCHES, TPN = run_inference(
        VAL_S1, POOL, TRANSLIT, CTX, predict, RULE, CONFIG['BLOCK_CAPS'], CONFIG['MAX_CAND_PER_S1'], CONFIG['CHUNK_S1'],
        'validation', s1_nt=_val_nt, pool_nt=PN)
    OUT_LABEL = 'VALIDATION-SPLIT output (test files were not found)'
print(f"=== {OUT_LABEL} ===")
MATCH_DF, CAND_DF, MATCHING_PATH, CANDIDATE_PATH = write_submission(SUB_IDS, SUB_CANDS, SUB_MATCHES, OUTPUT_DIR)
log(f"inference + writing finished in {fmt_secs(time.time() - T_INFER)}")

## 14. Submission validation

`validate_submission_files` re-implements every rule the scorer enforces (exact headers, at most two tab-separated columns, no quote characters, every test Source-1 id exactly once and no extra rows, only S2-/S3- ids that exist in the test files, no repeated ids inside a list, matches ⊆ candidates) and additionally runs the official `validate_submission.py` when it is found, pointing it at a directory whose file names are exactly `test_source1/2/3.tsv`.

In [ ]:
# =====================================================================================
# Cell 16 - submission validation (re-implemented rules + the official validator when available)
# =====================================================================================
def validate_submission_files(matching_path, candidate_path, s1_ids, pool_ids):
    """Every rule of the official validator, plus the id-existence and subset checks."""
    t0 = time.time()
    issues = []
    required = set(s1_ids); valid = set(pool_ids)
    parsed = {}
    for path, header, col in [(matching_path, 'source1_entity_id\tmatched_entity_ids', 'matched_entity_ids'),
                              (candidate_path, 'source1_entity_id\tcandidate_entity_ids', 'candidate_entity_ids')]:
        name = os.path.basename(path)
        if not os.path.isfile(path):
            issues.append(f"{name}: file not found"); continue
        mapping = {}; seen = set(); dup_rows = intra = wrong = unknown = bad_cols = quotes = 0
        with open(path, encoding='utf-8') as fh:
            head = fh.readline().rstrip('\n')
            if head != header:
                issues.append(f"{name}: header {head!r} != {header!r}")
            for line in fh:
                line = line.rstrip('\n')
                if not line:
                    continue
                if '"' in line:
                    quotes += 1
                parts = line.split('\t')
                if len(parts) > 2:
                    bad_cols += 1
                sid = parts[0]
                rest = parts[1] if len(parts) > 1 else ''
                if sid in seen:
                    dup_rows += 1
                seen.add(sid)
                ids = [x for x in rest.split(',') if x] if rest.strip() else []
                if len(ids) != len(set(ids)):
                    intra += 1
                for x in ids:
                    if not (x.startswith('S2-') or x.startswith('S3-')):
                        wrong += 1
                    elif x not in valid:
                        unknown += 1
                mapping[sid] = set(ids)
        missing = required - seen; extra = seen - required
        for cnt, msg in [(quotes, 'lines containing a quote character'), (bad_cols, 'rows with more than 2 tab-separated columns'),
                         (dup_rows, 'duplicate source1_entity_id rows'), (intra, f'{col} lists with repeated ids'),
                         (wrong, f'{col} ids without an S2-/S3- prefix'), (unknown, f'{col} ids not present in the test S2/S3 files'),
                         (len(missing), 'required Source-1 ids missing'), (len(extra), 'rows with ids not in test_source1')]:
            if cnt:
                issues.append(f"{name}: {cnt:,} {msg}")
        parsed[name] = mapping
        print(f"  {name}: {len(mapping):,} rows, {sum(1 for v in mapping.values() if not v):,} empty")
    m, c = parsed.get('matching_results.tsv'), parsed.get('candidate_pairs.tsv')
    if m is not None and c is not None:
        bad = sum(1 for k, v in m.items() if v - c.get(k, set()))
        if bad:
            issues.append(f"{bad:,} entities have matched ids that are not among their candidates")
    if issues:
        print("FAIL - issues found:")
        for i, msg in enumerate(issues, 1):
            print(f"  {i}. {msg}")
    else:
        print(f"PASS - both files satisfy every rule ({len(required):,} Source-1 entities, {len(valid):,} valid S2/S3 ids) "
              f"[{time.time() - t0:.1f}s]")
    return issues


_pool_ids_for_check = TPN.get('entity_id')
SUBMISSION_ISSUES = validate_submission_files(MATCHING_PATH, CANDIDATE_PATH, SUB_IDS, _pool_ids_for_check)
del _pool_ids_for_check
gc.collect()
assert not SUBMISSION_ISSUES, SUBMISSION_ISSUES

# official validator: it expects the literal file names test_source1/2/3.tsv inside --test-dir, so a directory
# of symlinks with those names is built when the discovered files carry an upload prefix
VALIDATOR = find_file('validate_submission.py')
if VALIDATOR and HAVE_TEST:
    test_dir = os.path.join(ARTIFACT_DIR, 'validator_test_dir')
    os.makedirs(test_dir, exist_ok=True)
    for i in (1, 2, 3):
        link = os.path.join(test_dir, f'test_source{i}.tsv')
        if os.path.lexists(link):
            os.remove(link)
        try:
            os.symlink(os.path.abspath(PATHS[f'test_source{i}.tsv']), link)
        except OSError:
            import shutil
            shutil.copyfile(PATHS[f'test_source{i}.tsv'], link)
    cmd = [sys.executable, VALIDATOR, '--matching', MATCHING_PATH, '--candidate', CANDIDATE_PATH, '--test-dir', test_dir]
    print('$ ' + ' '.join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout); print(res.stderr)
    print(f"official validator exit code: {res.returncode}")
    assert res.returncode == 0, "official validator reported issues"
else:
    print("official validate_submission.py not found or no test files -> skipped (the rules above were re-implemented)")

## 15. Artifacts and a stand-alone inference function

`artifacts/er_artifacts.joblib` holds the model, the best iteration, the decision rule, the transliteration map, the three TF-IDF vectorizers, the IDF table, the feature names, the block caps, the candidate cap and the configuration. `resolve_entities(s1_df, s2_df, s3_df)` rebuilds the feature context from these fields and returns the two result frames (optionally writing the files), demonstrated on the first 200 test Source-1 rows against a 50k-row slice of the test pool.

In [ ]:
# =====================================================================================
# Cell 17 - save model + preprocessing; stand-alone inference function
# =====================================================================================
ARTIFACTS = dict(model=MODEL, uses_lightgbm=HAVE_LGB, best_iteration=BEST_ITER, rule=RULE, translit_map=TRANSLIT,
                 tf_char=CTX.tf_char, tf_word=CTX.tf_word, tf_addr=CTX.tf_addr, idf=CTX.idf, idf_default=CTX.idf_default,
                 feature_names=FEATURE_NAMES, block_caps=CONFIG['BLOCK_CAPS'], max_cand_per_s1=CONFIG['MAX_CAND_PER_S1'],
                 config=CONFIG, validation=dict(macro_f05=MACRO_F05, precision=MICRO_P, recall=MICRO_R, n_entities=int(N_ENT_VA)))
ARTIFACT_PATH = os.path.join(ARTIFACT_DIR, 'er_artifacts.joblib')
joblib.dump(ARTIFACTS, ARTIFACT_PATH, compress=3)
log(f"artifacts saved: {ARTIFACT_PATH} ({os.path.getsize(ARTIFACT_PATH) / 1e6:.1f} MB)")


def resolve_entities(s1_df, s2_df, s3_df, artifacts=None, output_dir=None):
    """Stand-alone entry point: frames with entity_id / business_name / business_address / country ->
    (matching_results DataFrame, candidate_pairs DataFrame); files are written when output_dir is given."""
    art = artifacts or joblib.load(ARTIFACT_PATH)
    ctx = FeatureContext(artifacts=art)
    model, names, best = art['model'], art['feature_names'], art['best_iteration']
    if art['uses_lightgbm']:
        def pred(X):
            return model.predict(X[names], num_iteration=best)
    else:
        def pred(X):
            return model.predict_proba(X[names])[:, 1]
    pool = pd.concat([s2_df, s3_df], ignore_index=True)
    for df in (s1_df, pool):
        for c in REQUIRED_COLS:
            assert c in df.columns, f"missing column {c}"
    ids, cands, matches, _ = run_inference(s1_df.reset_index(drop=True), pool, art['translit_map'], ctx, pred, art['rule'],
                                           art['block_caps'], art['max_cand_per_s1'], CONFIG['CHUNK_S1'], 'resolve')
    seen = set(); rows = []
    for sid, c, m in zip(ids, cands, matches):
        if sid in seen:
            continue
        seen.add(sid); rows.append((sid, m, c))
    mres = pd.DataFrame(rows, columns=['source1_entity_id', 'matched_entity_ids', 'candidate_entity_ids'])
    cres = mres[['source1_entity_id', 'candidate_entity_ids']].copy()
    mres = mres[['source1_entity_id', 'matched_entity_ids']]
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        mres.to_csv(os.path.join(output_dir, 'matching_results.tsv'), sep='\t', index=False, quoting=csv.QUOTE_NONE, escapechar='\\', lineterminator='\n')
        cres.to_csv(os.path.join(output_dir, 'candidate_pairs.tsv'), sep='\t', index=False, quoting=csv.QUOTE_NONE, escapechar='\\', lineterminator='\n')
    return mres, cres


# demo: first 200 test Source-1 rows against a 50k-row slice of the test pool (validation frames when no test files)
_demo_s1 = (T1 if HAVE_TEST else S1.iloc[VAL_ROWS]).head(200).reset_index(drop=True)
_demo_pool = (TPOOL if HAVE_TEST else POOL).head(50_000)
_demo_s2 = _demo_pool[_demo_pool['entity_id'].str.startswith('S2')].reset_index(drop=True)
_demo_s3 = _demo_pool[_demo_pool['entity_id'].str.startswith('S3')].reset_index(drop=True)
DEMO_MATCHES, DEMO_CANDS = resolve_entities(_demo_s1, _demo_s2, _demo_s3, artifacts=joblib.load(ARTIFACT_PATH))
print(DEMO_MATCHES.head(10).to_string(index=False))
print(f"demo: {len(DEMO_MATCHES)} entities, {(DEMO_MATCHES['matched_entity_ids'] != '').sum()} with matches in the 50k-row slice")

## 16. Files generated

| file | path |
|---|---|
| final matches | `/kaggle/working/output/matching_results.tsv` |
| candidate set scored by the model | `/kaggle/working/output/candidate_pairs.tsv` |
| model + preprocessing | `/kaggle/working/artifacts/er_artifacts.joblib` |
| feature importance | `/kaggle/working/artifacts/feature_importance.csv` |
| threshold grid | `/kaggle/working/artifacts/threshold_grid.csv` |
| validation metrics | `/kaggle/working/artifacts/validation_metrics.json` |

(Outside Kaggle the same files are written under `./output` and `./artifacts`; the code cell below prints the resolved paths and sizes.)

### Cleaning → Normalization → Blocking → Candidate Features → ML Model → Thresholding → Final Matches → Submission Validation

**Cleaning.** Every name and address is stripped of the noise the sources inject — URL / phone / `(ID: …)` appendices, alias prefixes, honorifics, brackets, dotted abbreviations, apostrophes, punctuation, repeated words, legal tokens, `NULL`/`N/A`/`PO Box` fillers, unit designators and place words — without ever deleting non-ASCII text, overwriting digits or touching accents of non-Latin strings, because each of those shortcuts destroys real information in this data.

**Normalization.** Names become an ordered `name_norm` for similarity scoring and an order-free `name_key` (plus a digit-to-letter `leet_key`) for blocking; addresses become ordered / order-free token strings plus state, geo, city, house, street and postcode fields, using country-specific state tables only where they apply and open-set fallbacks everywhere else, so France works without any French table.

**Blocking.** Nine hashed key families with per-family caps are queried with sorted-array binary search; the union of the hits is ranked by the number and kind of agreeing families and cut at 50 candidates per Source-1 entity. That set is the candidate universe: it is scored by the model and written verbatim to `candidate_pairs.tsv`.

**Candidate features.** 77 float32 features per pair combine string similarities, TF-IDF cosines, token-set statistics, address agreement, name frequencies, the blocking hit pattern and entity-relative ranks; TF-IDF and IDF are fitted only on training-split text, and no feature encodes the country.

**ML model.** A LightGBM classifier learns from the blocked pairs of 120k training entities (hard negatives = the other blocked candidates) with early stopping on 40k held-out entities; the split is by entity, so no validation pair is ever seen in training.

**Thresholding.** Because the metric is a macro F0.5 over entities, the probability threshold (and an optional entity-relative margin) is chosen on the validation entities to maximise exactly that quantity, with singletons and never-blocked matches included.

**Final matches.** The test files are normalised with the saved dictionary, blocked, featurised and scored in chunks; the accepted candidates of every test Source-1 entity (including every French one) form `matching_results.tsv`, a subset of `candidate_pairs.tsv`.

**Submission validation.** Both files are checked against every scorer rule by the notebook and by the official validator, and the model plus all preprocessing state is saved for stand-alone use.

**Pinned requirements for the submission package:** `pandas>=2.2`, `numpy`, `scikit-learn`, `lightgbm`, `rapidfuzz>=3.6`, `pyarrow`, `joblib`, `matplotlib`.

In [ ]:
# =====================================================================================
# Cell 18 - files generated
# =====================================================================================
FILES = [os.path.join(OUTPUT_DIR, 'matching_results.tsv'), os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv'),
         os.path.join(ARTIFACT_DIR, 'er_artifacts.joblib'), os.path.join(ARTIFACT_DIR, 'feature_importance.csv'),
         os.path.join(ARTIFACT_DIR, 'threshold_grid.csv'), os.path.join(ARTIFACT_DIR, 'validation_metrics.json'),
         os.path.join(ARTIFACT_DIR, 'threshold_curve.png'), os.path.join(ARTIFACT_DIR, 'eda_charts.png')]
rows = []
for f in FILES:
    exists = os.path.isfile(f)
    rows.append(dict(file=os.path.basename(f), path=os.path.abspath(f),
                     size=(f"{os.path.getsize(f) / 1e6:.2f} MB" if exists else 'MISSING')))
FILES_TABLE = pd.DataFrame(rows)
print(FILES_TABLE.to_string(index=False))
print(f"\nsubmission content: {OUT_LABEL}")
print(f"validation macro F0.5 = {MACRO_F05:.5f} with rule {RULE} | total notebook time {fmt_secs(time.time() - T_NOTEBOOK_START)}")